<a href="https://colab.research.google.com/github/khaledBIM/khaledBIM/blob/main/Guide_BIMLIB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pyyaml python-docx cairosvg -q
print("✅ OK")

✅ OK


In [ ]:
import yaml, os, io, zipfile, textwrap, copy
import pandas as pd
from docx import Document
from docx.shared import Pt, RGBColor, Cm, Inches, Twips
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml.ns import qn
from docx.oxml import OxmlElement
from docx.enum.section import WD_SECTION
import datetime

# ════════════════════════════════════════════════════════════════════════════
#  FICHE D'IDENTITÉ — extraction dynamique des métadonnées (V21, data-driven)
# ════════════════════════════════════════════════════════════════════════════
def extraire_metadonnees(data, nom_famille):
    """
    Extrait les métadonnées du YAML pour la Fiche d'Identité — V21.
    Chaque champ possède un fallback intelligent : si la clé est absente du
    YAML, une valeur par défaut ou "(à compléter)" est affichée au lieu de
    provoquer une erreur. Cela permet d'enrichir progressivement les YAML.
    """
    discipline = str(data.get('discipline', '')).strip() or nom_famille[:3].upper()
    categorie = str(data.get('category', '')).strip() or "(à compléter)"
    version_revit = str(data.get('revit_version', '')).strip() or "2023"
    # Code Uniformat II = 2e segment du nom de famille
    #   ex. "SIQ-E2010-Caisson" -> "E2010". Fallback sur les clés YAML.
    segments_nom = str(nom_famille).split('-')
    uniformat_depuis_nom = segments_nom[1].strip() if len(segments_nom) > 1 else ''
    uniformat_code = (
        uniformat_depuis_nom or
        str(data.get('uniformat_code', '')).strip() or
        str(data.get('uniformat_ii', '')).strip() or
        str(data.get('uniformat', '')).strip() or
        "(à compléter)"
    )
    version_ifc = str(data.get('ifc_version', '')).strip() or "(à compléter)"
    ifc_export = str(data.get('ifc_export_type', '')).strip() or "(à compléter)"
    ifc_type = str(data.get('ifc_type', '')).strip() or "(à compléter)"

    glossaire = data.get('glossaire', {})
    if isinstance(glossaire, dict):
        glossaire_str = " / ".join(
            f"{k.upper()} : {v}" for k, v in glossaire.items()
        ) or "(à compléter)"
    else:
        glossaire_str = str(glossaire).strip() or "(à compléter)"

    reference = str(data.get('reference_produit', '')).strip() or ""
    if not reference:
        reference = str(data.get('reference', '')).strip() or "(Coller référence fabricant)"

    return {
        'nom_objet': nom_famille,
        'discipline': discipline,
        'categorie': categorie,
        'version_revit': version_revit,
        'uniformat_code': uniformat_code,
        'version_ifc': version_ifc,
        'ifc_export_type': ifc_export,
        'ifc_type_predefini': ifc_type,
        'glossaire': glossaire_str,
        'reference_produit': reference,
    }

# ════════════════════════════════════════════════════════════════════════════
#  LECTURE DU FICHIER EXCEL — hiérarchie multi-niveaux
# ════════════════════════════════════════════════════════════════════════════
def lire_imbriquees_depuis_yaml(data):
    """
    Remplace lire_excel() — lit nested_families[] du YAML (produit par Cellule 1 V6).
    Retourne exactement le même format de liste qu'attendait l'ancien code Excel :
      nom, type, categorie, niveau, parent, enfants, types (liste), parameters (dict)
    Récursif : supporte N niveaux d'imbrication.
    Exclut les catégories de symboles d'annotation.
    """
    CATEGORIES_EXCLUES = {
        "Symboles d'annotation", "Symboles annotatifs",
        "Extrémités des lignes de coupe", "Marqueurs de niveau",
    }

    toutes      = {}   # nom → info dict
    ordonnees   = []   # résultat final DFS

    def parcourir(noeud, parent_nom, niveau):
        nom = str(noeud.get('nom', '')).strip()
        if not nom:
            return
        cat = str(noeud.get('categorie', 'Famille imbriquée')).strip() or 'Famille imbriquée'
        if cat in CATEGORIES_EXCLUES:
            return

        types_list  = [str(t) for t in (noeud.get('types', []) or [])]
        type_val    = types_list[0] if types_list else ''
        params_dict = noeud.get('parameters', {}) or {}

        info = {
            'nom':        nom,
            'type':       type_val,
            'types':      types_list,
            'categorie':  cat,
            'is_shared':  bool(noeud.get('is_shared', False)),
            'niveau':     niveau,
            'parent':     parent_nom,
            'enfants':    [],
            'parameters': params_dict,
        }

        if nom not in toutes:
            toutes[nom] = info
        if parent_nom and parent_nom in toutes:
            if nom not in toutes[parent_nom]['enfants']:
                toutes[parent_nom]['enfants'].append(nom)

        ordonnees.append(info)

        for enfant_noeud in (noeud.get('nested_families', []) or []):
            parcourir(enfant_noeud, nom, niveau + 1)

    for noeud_racine in (data.get('nested_families', []) or []):
        parcourir(noeud_racine, None, 1)

    print(f"    🔗 YAML : {len(ordonnees)} famille(s) imbriquée(s) (tous niveaux)")
    return ordonnees


def lire_excel_compat(excel_path):
    """
    Fallback pour compatibilité ascendante : lit un Excel de familles imbriquées.
    Utilisé uniquement si le YAML ne contient pas nested_families[].
    """
    df = pd.read_excel(excel_path)
    df.columns = [c.strip() for c in df.columns]
    col_nom  = next((c for c in df.columns if 'famille' in c.lower() and 'nom' in c.lower()), df.columns[0])
    col_type = next((c for c in df.columns if 'type' in c.lower()), df.columns[1])
    col_cat  = next((c for c in df.columns if 'catég' in c.lower() or 'categ' in c.lower()), df.columns[2])
    col_id   = next((c for c in df.columns if 'id' in c.lower()), None)
    cols_fixes = {col_nom, col_type, col_cat, col_id} if col_id else {col_nom, col_type, col_cat}
    cols_niv_sup = [c for c in df.columns if c not in cols_fixes]
    toutes = {}
    for _, row in df.iterrows():
        nom_n1 = str(row[col_nom]).strip()
        cat_n1 = str(row[col_cat]).strip()
        if cat_n1 in ["Symboles d'annotation"]: continue
        if nom_n1 not in toutes:
            toutes[nom_n1] = {
                'nom': nom_n1, 'type': str(row[col_type]).strip(),
                'types': [], 'categorie': cat_n1,
                'id': str(row[col_id]).strip() if col_id else '',
                'niveau': 1, 'parent': None, 'enfants': [], 'parameters': {},
            }
        parent_courant = nom_n1
        for depth, col_niv in enumerate(cols_niv_sup, start=2):
            val = row[col_niv]
            if pd.isna(val) or str(val).strip() in ('', 'nan', 'NaN'): break
            nom_enfant = str(val).strip()
            if nom_enfant not in toutes:
                toutes[nom_enfant] = {
                    'nom': nom_enfant, 'type': '', 'types': [],
                    'categorie': 'Famille imbriquée', 'id': '',
                    'niveau': depth, 'parent': parent_courant, 'enfants': [], 'parameters': {},
                }
            if nom_enfant not in toutes[parent_courant]['enfants']:
                toutes[parent_courant]['enfants'].append(nom_enfant)
            parent_courant = nom_enfant
    def dfs(nom, visited=None):
        if visited is None: visited = set()
        if nom in visited: return []
        visited.add(nom)
        result = [toutes[nom]]
        for enfant in toutes[nom].get('enfants', []):
            if enfant in toutes: result.extend(dfs(enfant, visited))
        return result
    familles_ordonnees = []
    visited_global = set()
    for nom, info in toutes.items():
        if info['niveau'] == 1:
            familles_ordonnees.extend(dfs(nom, visited_global))
    return familles_ordonnees

# ════════════════════════════════════════════════════════════════════════════
#  TRADUCTIONS
# ════════════════════════════════════════════════════════════════════════════
GROUPES_FR = {
    "visibility":   "Visibilité",
    "dimensions":   "Cotes",
    "constraints":  "Contraintes",
    "identityData": "Données d'identification",
    "text":         "Texte",
    "data":         "Données",
    "ifc":          "IFC",
    "materials":    "Matériaux et finitions",
    "electrical":   "Électricité",
    "mechanical":   "Mécanique",
    "structural":   "Structure",
    "general":      "Général",
    "construction": "Construction",
}
TYPES_FR = {
    "spec.bool":           "Oui / Non",
    "spec.string":         "Texte",
    "spec.int64":          "Entier",
    "aec:length":          "Variable",
    "aec:number":          "Nombre",
    "measurable:currency": "Devise",
    "reference:image":     "Image",
    "string:url":          "URL",
}

def traduire_groupe(g):
    if not g: return "Général"
    key = g.split(":")[-1].split("-")[0]
    return GROUPES_FR.get(key, key.capitalize())

def traduire_type(t):
    if not t: return "Oui / Non"
    for k, v in TYPES_FR.items():
        if k in t: return v
    return "Texte"

# ════════════════════════════════════════════════════════════════════════════
#  HELPERS WORD
# ════════════════════════════════════════════════════════════════════════════
def couleur_cellule(cell, hex_color):
    tcPr = cell._tc.get_or_add_tcPr()
    shd = OxmlElement('w:shd')
    shd.set(qn('w:fill'), hex_color)
    shd.set(qn('w:val'), 'clear')
    tcPr.append(shd)

def style_run(run, bold=False, size=9, color=None, italic=False):
    """Style un run avec Parisine Office (police RATP) en priorité, Arial en fallback."""
    run.bold = bold
    run.font.size = Pt(size)
    # Police principale : Parisine Office (RATP), fallback Arial
    rPr = run._r.get_or_add_rPr()
    old_fonts = rPr.find(qn('w:rFonts'))
    if old_fonts is not None:
        rPr.remove(old_fonts)
    rFonts = OxmlElement('w:rFonts')
    rFonts.set(qn('w:ascii'), 'Parisine Office')
    rFonts.set(qn('w:hAnsi'), 'Parisine Office')
    rPr.insert(0, rFonts)
    run.italic = italic
    if color:
        r, g, b = int(color[0:2],16), int(color[2:4],16), int(color[4:6],16)
        run.font.color.rgb = RGBColor(r, g, b)

def entete_tableau(table, labels, couleur=None):
    """
    En-tête tableau calqué sur le nouveau template RATP :
    - Fond blanc (pas de fond coloré)
    - Texte noir bold, Parisine Office, 9pt (sz=18)
    - Centré, bordure bottom visible pour séparer de données
    """
    row = table.rows[0]
    for i, label in enumerate(labels):
        if i >= len(row.cells): break
        cell = row.cells[i]
        cell.text = label
        # Fond blanc — pas de couleur de fond
        couleur_cellule(cell, "FFFFFF")
        p = cell.paragraphs[0]
        p.alignment = WD_ALIGN_PARAGRAPH.CENTER
        run = p.runs[0]
        # sz=18 = 9pt bold noir — calqué sur le nouveau template
        style_run(run, bold=True, size=9, color="000000")
        rPr = run._r.get_or_add_rPr()
        rf_old = rPr.find(qn('w:rFonts'))
        if rf_old is not None: rPr.remove(rf_old)
        rf = OxmlElement('w:rFonts')
        for a in ['ascii', 'hAnsi', 'eastAsia', 'cs']:
            rf.set(qn(f'w:{a}'), 'Parisine Office')
        rPr.insert(0, rf)
        # Bordure bottom pour séparer entête des données
        tcPr = cell._tc.get_or_add_tcPr()
        tcBdr = OxmlElement('w:tcBdr')
        bottom = OxmlElement('w:bottom')
        bottom.set(qn('w:val'), 'single')
        bottom.set(qn('w:sz'), '4')
        bottom.set(qn('w:space'), '0')
        bottom.set(qn('w:color'), '000000')
        tcBdr.append(bottom)
        tcPr.append(tcBdr)

def cellule_donnee(cell, texte, pair=False, center=False, bold=False, color=None):
    """
    Cellule de données calquée sur le fichier de référence :
    - Fond blanc / gris très clair F5F5F5 en alternance
    - Parisine Office, taille 8pt (sz=16)
    - Paramètres non-modifiables : rouge gras
    """
    couleur_cellule(cell, "FFFFFF")  # fond blanc seul — nouveau template
    p = cell.paragraphs[0]
    p.clear()
    run = p.add_run(str(texte) if texte is not None else "—")
    # sz=16 = 8pt — calqué sur le fichier de référence
    style_run(run, size=8, bold=bold, color=color)
    # Forcer Parisine Office
    rPr = run._r.get_or_add_rPr()
    rf_old = rPr.find(qn('w:rFonts'))
    if rf_old is not None: rPr.remove(rf_old)
    rf = OxmlElement('w:rFonts')
    for a in ['ascii', 'hAnsi', 'eastAsia', 'cs']:
        rf.set(qn(f'w:{a}'), 'Parisine Office')
    rPr.insert(0, rf)
    if center:
        p.alignment = WD_ALIGN_PARAGRAPH.CENTER

def set_col_widths(table, widths_cm):
    tbl = table._tbl
    tblPr = tbl.find(qn('w:tblPr'))
    if tblPr is None:
        tblPr = OxmlElement('w:tblPr')
        tbl.insert(0, tblPr)
    # tblLayout doit venir après tblW — on supprime l'éventuel existant et on réinsère
    old_layout = tblPr.find(qn('w:tblLayout'))
    if old_layout is not None:
        tblPr.remove(old_layout)
    tblLayout = OxmlElement('w:tblLayout')
    tblLayout.set(qn('w:type'), 'fixed')
    # Insérer après tblW (premier enfant) si possible
    tblW_el = tblPr.find(qn('w:tblW'))
    if tblW_el is not None:
        tblW_el.addnext(tblLayout)
    else:
        tblPr.insert(0, tblLayout)
    for row in table.rows:
        for i, cell in enumerate(row.cells):
            if i < len(widths_cm):
                tc = cell._tc
                tcPr = tc.get_or_add_tcPr()
                # Supprimer l'éventuel tcW existant
                old_tcW = tcPr.find(qn('w:tcW'))
                if old_tcW is not None:
                    tcPr.remove(old_tcW)
                tcW = OxmlElement('w:tcW')
                twips = int(widths_cm[i] * 567)
                tcW.set(qn('w:w'), str(twips))
                tcW.set(qn('w:type'), 'dxa')
                # tcW doit être en première position dans tcPr
                tcPr.insert(0, tcW)

def set_keep_with_next(paragraph):
    """Empêche un saut de page entre ce paragraphe et le suivant."""
    pPr = paragraph._p.get_or_add_pPr()
    keepNext = OxmlElement('w:keepNext')
    pPr.append(keepNext)

def set_keep_together(paragraph):
    """Empêche un saut de page à l'intérieur de ce paragraphe."""
    pPr = paragraph._p.get_or_add_pPr()
    keepLines = OxmlElement('w:keepLines')
    pPr.append(keepLines)

def set_page_break_before(paragraph):
    """Ajoute un saut de page avant ce paragraphe."""
    pPr = paragraph._p.get_or_add_pPr()
    pageBreakBefore = OxmlElement('w:pageBreakBefore')
    pPr.append(pageBreakBefore)

# Paramètres built-in Revit à exclure des tableaux BIM
PARAMS_BUILTIN_EXCLUS = {
    # Coûts
    "coût", "cout", "cost",
    # Assemblage
    "code d'assemblage", "code assemblage", "assembly code",
    "description d'assemblage", "assembly description",
    # Identité / type
    "commentaires", "comments",
    "commentaires du type", "type comments",
    "description",
    "url", "url du type", "type url",
    "keynote", "note d'identification", "note d'identification du type",
    "marque", "mark",
    "modèle", "modele", "model",
    "fabricant", "manufacturer",
    "nom de type", "type name",
    "nom de famille et de type", "famille et type", "family and type",
    "description du type", "type description",
    "image", "image du type", "image de type", "type image",
    # IFC natif
    "type ifc", "ifc type", "guid ifc", "ifc guid",
    "exporter le type au format ifc sous",
    "type prédéfini d'ifc", "ifc predefined type",
    # Annotation / affichage natif
    "utiliser l'échelle d'annotations", "use annotation scale",
    # Matériau natif
    "matériau pour modèle", "materiau pour modele", "material for model",
    # Données structurelles / phases
    "phase de création", "phase created",
    "phase de destruction", "phase demolished",
    # Placement / contraintes natives
    "niveau", "level",
    "élévation par défaut", "default elevation",
    "se déplace avec les quadrillages", "moves with grids",
    "se déplace avec les éléments proches", "moves with nearby elements",
    "toujours vertical", "always vertical",
    "basé sur l'espace de travail", "work plane-based",
    "hôte", "host",
}

# Préfixes de paramètres MÉTIER / PROJET : jamais considérés comme built-in
PREFIXES_CUSTOM = ("ratp_", "as_", "cf_")

def est_builtin(name):
    """
    Retourne True si le paramètre est un built-in Revit natif à exclure
    du tableau des paramètres. Les paramètres métier (RATP_/AS_/CF_) sont
    toujours conservés, même s'ils contiennent un motif ci-dessous.
    """
    nl = name.lower().strip()
    # Protection : un paramètre métier/projet n'est jamais built-in
    if nl.startswith(PREFIXES_CUSTOM):
        return False
    # Exclusion exacte
    if nl in PARAMS_BUILTIN_EXCLUS:
        return True
    # Exclusion par contenu (motifs sûrs, absents des noms métier)
    builtin_patterns = [
        "assembl",      # code / description d'assemblage
        "keynote",
        "ifc",          # IfcName, IfcDescription, IfcExportAs...
        "omniclass",
        "uniformat",    # déjà géré dans la fiche d'identité
    ]
    for pat in builtin_patterns:
        if pat in nl:
            return True
    return False

def est_parametre_fixe(info):
    """
    Retourne True si le paramètre ne doit PAS être modifié (Modifier = "Non") :
    - paramètre de TYPE (pas d'occurrence) : instance=False
    - paramètre piloté par une formule (formula non vide)
    - paramètre grisé / verrouillé en lecture seule (reporting=True)
    Sinon (occurrence, sans formule, non-reporting) → modifiable (Modifier = "Oui").
    """
    formule = str(info.get('formula', '') or '').strip()
    instance = bool(info.get('instance', True))  # True = occurrence par défaut
    reporting = bool(info.get('reporting', False))  # True = grisé/verrouillé dans Revit
    est_type = not instance
    a_formule = bool(formule) and formule not in ('-', 'None', '')
    return est_type or a_formule or reporting

def grouper_parametres(parametres):
    groupes = {}
    for name, info in parametres.items():
        # Exclure les paramètres built-in Revit
        if est_builtin(name):
            continue
        grp = traduire_groupe(info.get('group', '') or '')
        if "VIS_" in name or "VISIBILITE" in name.upper():
            grp = "Visibilité"
        elif any(k in name.upper() for k in ["DIM_","PROFONDEUR","HAUTEUR","LARGEUR","LONGUEUR","NGF","COORD","DIAM"]):
            grp = "Cotes"
        elif grp == "Général" and (name.startswith("RATP_") or name.startswith("AS_")):
            grp = "Autres Paramètres"
        groupes.setdefault(grp, []).append((name, info))
    ordre = ["Visibilité","Cotes","Contraintes","Autres Paramètres"]
    reste = [k for k in groupes if k not in ordre]
    return {k: groupes[k] for k in ordre + reste if k in groupes}

# ════════════════════════════════════════════════════════════════════════════
#  HELPERS EN-TÊTE / PIED DE PAGE (fidèles au template RATP)
# ════════════════════════════════════════════════════════════════════════════
def ajouter_bordure_cellule(tcPr, top=None, bottom=None, left=None, right=None):
    """Ajoute des bordures — tcPr est l'élément w:tcPr directement.
    Note : dans OOXML strict, w:left→w:start et w:right→w:end dans tcBorders."""
    tcBorders = OxmlElement('w:tcBorders')
    # Ordre OOXML : top, start, bottom, end
    for side_xml, val in [('top', top), ('start', left), ('bottom', bottom), ('end', right)]:
        if val == 'nil':
            el = OxmlElement(f'w:{side_xml}'); el.set(qn('w:val'), 'nil'); tcBorders.append(el)
        elif val == 'single':
            el = OxmlElement(f'w:{side_xml}')
            el.set(qn('w:val'), 'single'); el.set(qn('w:sz'), '4')
            el.set(qn('w:space'), '0'); el.set(qn('w:color'), '000000')
            tcBorders.append(el)
    tcPr.append(tcBorders)

def creer_run_field(instrText):
    """Crée un champ Word (PAGE, NUMPAGES, etc.) sous forme de liste de runs XML."""
    runs = []
    for ftype, text in [('begin', None), (None, instrText), ('separate', None), ('end', None)]:
        r = OxmlElement('w:r')
        rPr = OxmlElement('w:rPr')
        color = OxmlElement('w:color'); color.set(qn('w:val'), '000000')
        sz = OxmlElement('w:sz'); sz.set(qn('w:val'), '18')
        szCs = OxmlElement('w:szCs'); szCs.set(qn('w:val'), '18')
        rPr.append(color); rPr.append(sz); rPr.append(szCs)
        r.append(rPr)
        if ftype:
            fc = OxmlElement('w:fldChar'); fc.set(qn('w:fldCharType'), ftype); r.append(fc)
        else:
            it = OxmlElement('w:instrText')
            it.set(qn('xml:space'), 'preserve')
            it.text = f' {instrText} '
            r.append(it)
        runs.append(r)
    return runs

def ajouter_entete_ratp(section, nom_guide):
    """
    En-tête RATP : tableau 2 colonnes
    Ligne 1 : "Département des systèmes d'information..." | [vide]
    Ligne 2 : "Programme BIM RATP ... <nom_guide>" (bordures haut/bas) | [vide]
    Ligne 3 : "Réf. : ... Type de diffusion : Accès restreint" (bordures) | [vide]
    """
    header = section.header
    # Vider le contenu existant
    for p in header.paragraphs:
        p._element.getparent().remove(p._element)

    # Paragraphe initial vide
    p0 = OxmlElement('w:p')
    pPr0 = OxmlElement('w:pPr')
    spacing0 = OxmlElement('w:spacing'); spacing0.set(qn('w:line'), '276'); spacing0.set(qn('w:lineRule'), 'auto')
    jc0 = OxmlElement('w:jc'); jc0.set(qn('w:val'), 'left')
    pPr0.append(spacing0); pPr0.append(jc0)
    p0.append(pPr0)
    header._element.append(p0)

    # Tableau en-tête
    tbl = OxmlElement('w:tbl')

    # tblPr
    tblPr = OxmlElement('w:tblPr')
    tblW = OxmlElement('w:tblW'); tblW.set(qn('w:w'), '8927'); tblW.set(qn('w:type'), 'dxa')
    tblInd = OxmlElement('w:tblInd'); tblInd.set(qn('w:w'), '-38'); tblInd.set(qn('w:type'), 'dxa')
    tblLayout = OxmlElement('w:tblLayout'); tblLayout.set(qn('w:type'), 'fixed')
    tblCellMar = OxmlElement('w:tblCellMar')
    start_m = OxmlElement('w:start'); start_m.set(qn('w:w'), '70'); start_m.set(qn('w:type'), 'dxa')
    end_m = OxmlElement('w:end'); end_m.set(qn('w:w'), '70'); end_m.set(qn('w:type'), 'dxa')
    tblCellMar.append(start_m); tblCellMar.append(end_m)
    for el in [tblW, tblInd, tblLayout, tblCellMar]:
        tblPr.append(el)
    tbl.append(tblPr)

    # tblGrid
    tblGrid = OxmlElement('w:tblGrid')
    for w in [6000, 2927]:
        gc = OxmlElement('w:gridCol'); gc.set(qn('w:w'), str(w)); tblGrid.append(gc)
    tbl.append(tblGrid)

    def make_row_entete(texte_col1, bold=False, avec_bordures=False, hauteur=None):
        tr = OxmlElement('w:tr')
        if hauteur:
            trPr = OxmlElement('w:trPr')
            trH = OxmlElement('w:trHeight'); trH.set(qn('w:val'), str(hauteur))
            trPr.append(trH); tr.append(trPr)

        # Cellule 1
        tc1 = OxmlElement('w:tc')
        tcPr1 = OxmlElement('w:tcPr')
        tcW1 = OxmlElement('w:tcW'); tcW1.set(qn('w:w'), '6000'); tcW1.set(qn('w:type'), 'dxa')
        tcPr1.append(tcW1)
        if avec_bordures:
            ajouter_bordure_cellule(tcPr1, top='single', bottom='single', left='nil', right='nil')
        else:
            ajouter_bordure_cellule(tcPr1, top='nil', bottom='nil', left='nil', right='nil')
        vAlign1 = OxmlElement('w:vAlign'); vAlign1.set(qn('w:val'), 'center' if avec_bordures else 'bottom')
        tcPr1.append(vAlign1)
        tc1.append(tcPr1)

        p1 = OxmlElement('w:p')
        pPr1 = OxmlElement('w:pPr')
        spacing1 = OxmlElement('w:spacing'); spacing1.set(qn('w:before'), '60'); spacing1.set(qn('w:after'), '60')
        jc1 = OxmlElement('w:jc'); jc1.set(qn('w:val'), 'left')
        pPr1.append(spacing1); pPr1.append(jc1)
        p1.append(pPr1)

        if isinstance(texte_col1, list):
            for i, ligne in enumerate(texte_col1):
                r = OxmlElement('w:r')
                rPr = OxmlElement('w:rPr')
                if bold:
                    b = OxmlElement('w:b'); rPr.append(b)
                color = OxmlElement('w:color'); color.set(qn('w:val'), '000000')
                sz = OxmlElement('w:sz'); sz.set(qn('w:val'), '18' if not bold else '28')
                szCs = OxmlElement('w:szCs'); szCs.set(qn('w:val'), '18' if not bold else '28')
                rPr.append(color); rPr.append(sz); rPr.append(szCs)
                r.append(rPr)
                if i > 0:
                    br = OxmlElement('w:br'); r.append(br)
                t = OxmlElement('w:t'); t.text = ligne
                if i > 0: t.set(qn('xml:space'), 'preserve')
                r.append(t)
                p1.append(r)
        else:
            r = OxmlElement('w:r')
            rPr = OxmlElement('w:rPr')
            if bold:
                b = OxmlElement('w:b'); rPr.append(b)
            color = OxmlElement('w:color'); color.set(qn('w:val'), '000000')
            sz = OxmlElement('w:sz'); sz.set(qn('w:val'), '18' if not bold else '28')
            szCs = OxmlElement('w:szCs'); szCs.set(qn('w:val'), '18' if not bold else '28')
            rPr.append(color); rPr.append(sz); rPr.append(szCs)
            r.append(rPr)
            t = OxmlElement('w:t'); t.text = texte_col1; r.append(t)
            p1.append(r)
        tc1.append(p1)

        # Cellule 2 (vide)
        tc2 = OxmlElement('w:tc')
        tcPr2 = OxmlElement('w:tcPr')
        tcW2 = OxmlElement('w:tcW'); tcW2.set(qn('w:w'), '2927'); tcW2.set(qn('w:type'), 'dxa')
        tcPr2.append(tcW2)
        if avec_bordures:
            ajouter_bordure_cellule(tcPr2, top='single', bottom='single', left='nil', right='nil')
        else:
            ajouter_bordure_cellule(tcPr2, top='nil', bottom='nil', left='nil', right='nil')
        tc2.append(tcPr2)
        # Ajouter le logo RATP dans la cellule de droite (première ligne seulement)
        p2 = OxmlElement('w:p')
        pPr2_logo = OxmlElement('w:pPr')
        jc2_logo = OxmlElement('w:jc'); jc2_logo.set(qn('w:val'), 'right')
        pPr2_logo.append(jc2_logo)
        p2.append(pPr2_logo)
        tc2.append(p2)

        tr.append(tc1); tr.append(tc2)
        return tr

    # Ligne 1 : département
    tbl.append(make_row_entete(
        ["Département des systèmes d\u2019information et télécommunications",
         "Systèmes d\u2019Information, Infrastructure et Production"],
        bold=False, avec_bordures=False, hauteur=1257
    ))
    # Ligne 2 : titre guide (avec bordures)
    tbl.append(make_row_entete(
        f"Guide d\u2019utilisation de l\u2019objet BIM : {nom_guide}",
        bold=True, avec_bordures=True
    ))
    # Ligne 3 : référence (avec bordures)
    tbl.append(make_row_entete(
        "Type de diffusion : Diffusion interne",
        bold=False, avec_bordures=True
    ))

    header._element.append(tbl)

    # Paragraphe final vide
    p_fin = OxmlElement('w:p')
    pPr_fin = OxmlElement('w:pPr')
    tabs_fin = OxmlElement('w:tabs')
    tab1 = OxmlElement('w:tab'); tab1.set(qn('w:val'), 'center'); tab1.set(qn('w:pos'), '4536')
    tab2 = OxmlElement('w:tab'); tab2.set(qn('w:val'), 'right'); tab2.set(qn('w:pos'), '9072')
    tabs_fin.append(tab1); tabs_fin.append(tab2)
    pPr_fin.append(tabs_fin)
    p_fin.append(pPr_fin)
    header._element.append(p_fin)


def ajouter_pied_page_ratp(section):
    """
    Pied de page RATP : tableau 3 colonnes + mention copyright
    Col 1 : "Guide d'utilisation des familles d'objets"
    Col 2 : "Type de diffusion : Diffusion interne"
    Col 3 : PAGE / NUMPAGES
    Suivi de : © Ce document est la propriété de la RATP...
    """
    footer = section.footer
    for p in footer.paragraphs:
        p._element.getparent().remove(p._element)

    # Paragraphe vide initial
    p0 = OxmlElement('w:p'); footer._element.append(p0)

    # Tableau pied de page
    tbl = OxmlElement('w:tbl')
    tblPr = OxmlElement('w:tblPr')
    tblW = OxmlElement('w:tblW'); tblW.set(qn('w:w'), '8789'); tblW.set(qn('w:type'), 'dxa')
    jc = OxmlElement('w:jc'); jc.set(qn('w:val'), 'center')
    tblLayout = OxmlElement('w:tblLayout'); tblLayout.set(qn('w:type'), 'fixed')
    tblCellMar = OxmlElement('w:tblCellMar')
    left_m = OxmlElement('w:start'); left_m.set(qn('w:w'), '70'); left_m.set(qn('w:type'), 'dxa')
    right_m = OxmlElement('w:end'); right_m.set(qn('w:w'), '70'); right_m.set(qn('w:type'), 'dxa')
    tblCellMar.append(left_m); tblCellMar.append(right_m)
    for el in [tblW, jc, tblLayout, tblCellMar]:
        tblPr.append(el)
    tbl.append(tblPr)

    tblGrid = OxmlElement('w:tblGrid')
    for w in [3681, 4394, 714]:
        gc = OxmlElement('w:gridCol'); gc.set(qn('w:w'), str(w)); tblGrid.append(gc)
    tbl.append(tblGrid)

    tr = OxmlElement('w:tr')
    trPr = OxmlElement('w:trPr')
    trjc = OxmlElement('w:jc'); trjc.set(qn('w:val'), 'center')
    trPr.append(trjc); tr.append(trPr)

    def pied_cellule(texte, largeur, center=False, avec_champ=False):
        tc = OxmlElement('w:tc')
        tcPr = OxmlElement('w:tcPr')
        tcW = OxmlElement('w:tcW'); tcW.set(qn('w:w'), str(largeur)); tcW.set(qn('w:type'), 'dxa')
        tcPr.append(tcW)
        ajouter_bordure_cellule(tcPr, top='single', bottom='single', left='single', right='single')
        if center:
            vAlign = OxmlElement('w:vAlign'); vAlign.set(qn('w:val'), 'center'); tcPr.append(vAlign)
        tc.append(tcPr)

        p = OxmlElement('w:p')
        pPr = OxmlElement('w:pPr')
        spacing = OxmlElement('w:spacing'); spacing.set(qn('w:before'), '120'); spacing.set(qn('w:after'), '120')
        pPr.append(spacing)
        if center:
            jc_p = OxmlElement('w:jc'); jc_p.set(qn('w:val'), 'center'); pPr.append(jc_p)
        p.append(pPr)

        if avec_champ:
            # PAGE / NUMPAGES
            for field_run in creer_run_field('PAGE'):
                p.append(field_run)
            r_sep = OxmlElement('w:r')
            rPr_sep = OxmlElement('w:rPr')
            c = OxmlElement('w:color'); c.set(qn('w:val'), '000000')
            s = OxmlElement('w:sz'); s.set(qn('w:val'), '18')
            sc = OxmlElement('w:szCs'); sc.set(qn('w:val'), '18')
            rPr_sep.append(c); rPr_sep.append(s); rPr_sep.append(sc)
            r_sep.append(rPr_sep)
            t_sep = OxmlElement('w:t'); t_sep.set(qn('xml:space'), 'preserve'); t_sep.text = ' / '
            r_sep.append(t_sep); p.append(r_sep)
            for field_run in creer_run_field('NUMPAGES'):
                p.append(field_run)
        else:
            r = OxmlElement('w:r')
            rPr = OxmlElement('w:rPr')
            color = OxmlElement('w:color'); color.set(qn('w:val'), '000000')
            sz = OxmlElement('w:sz'); sz.set(qn('w:val'), '18')
            szCs = OxmlElement('w:szCs'); szCs.set(qn('w:val'), '18')
            rPr.append(color); rPr.append(sz); rPr.append(szCs)
            r.append(rPr)
            t = OxmlElement('w:t'); t.set(qn('xml:space'), 'preserve'); t.text = texte
            r.append(t); p.append(r)

        tc.append(p)
        return tc

    tr.append(pied_cellule("Guide d\u2019utilisation des familles d\u2019objets", 3681))
    tr.append(pied_cellule("Type de diffusion : Diffusion interne", 4394))
    tr.append(pied_cellule("", 714, center=True, avec_champ=True))
    tbl.append(tr)
    footer._element.append(tbl)

    # Mention copyright
    p_copy = OxmlElement('w:p')
    pPr_copy = OxmlElement('w:pPr')
    tabs_copy = OxmlElement('w:tabs')
    tab1 = OxmlElement('w:tab'); tab1.set(qn('w:val'), 'center'); tab1.set(qn('w:pos'), '4536')
    tab2 = OxmlElement('w:tab'); tab2.set(qn('w:val'), 'right'); tab2.set(qn('w:pos'), '9072')
    tabs_copy.append(tab1); tabs_copy.append(tab2)
    pPr_copy.append(tabs_copy)
    p_copy.append(pPr_copy)

    r_copy1 = OxmlElement('w:r')
    rPr_c1 = OxmlElement('w:rPr')
    b1 = OxmlElement('w:b'); i1 = OxmlElement('w:i')
    color1 = OxmlElement('w:color'); color1.set(qn('w:val'), '000000')
    rPr_c1.append(b1); rPr_c1.append(i1); rPr_c1.append(color1)
    r_copy1.append(rPr_c1)
    t_c1 = OxmlElement('w:t'); t_c1.set(qn('xml:space'), 'preserve'); t_c1.text = '\u00a9 '
    r_copy1.append(t_c1)

    r_copy2 = OxmlElement('w:r')
    rPr_c2 = OxmlElement('w:rPr')
    color2 = OxmlElement('w:color'); color2.set(qn('w:val'), '000000')
    sz2 = OxmlElement('w:sz'); sz2.set(qn('w:val'), '18')
    szCs2 = OxmlElement('w:szCs'); szCs2.set(qn('w:val'), '18')
    rPr_c2.append(color2); rPr_c2.append(sz2); rPr_c2.append(szCs2)
    r_copy2.append(rPr_c2)
    t_c2 = OxmlElement('w:t')
    t_c2.text = 'Ce document est la propri\u00e9t\u00e9 de la RATP. Il ne peut \u00eatre reproduit, m\u00eame partiellement sans son autorisation'
    r_copy2.append(t_c2)

    p_copy.append(r_copy1); p_copy.append(r_copy2)
    footer._element.append(p_copy)

    # Paragraphe final
    p_fin = OxmlElement('w:p'); footer._element.append(p_fin)


# ════════════════════════════════════════════════════════════════════════════
#  PAGE DE GARDE (tableau BIM RATP : validation + suivi)
# ════════════════════════════════════════════════════════════════════════════
# Base64 du logo RATP Infrastructures (intégré directement)
LOGO_RATP_B64 = "..."

def ajouter_page_garde(doc, nom, date_str=None):
    """
    Page de garde V5 — fidèle au template RATP (260529_Template_mise_au_propre.docx) :
    Police principale : Parisine Office (avec fallback Arial)
    - Logo RATP Infrastructures (en-tête tableau 3 cols, logo à droite)
    - Programme BIM / RATP Infrastructures / DI3D_ISI_BIM (style BIM-CEB-TitreDépartement, 12pt)
    - Bandeau bleu nuit (0A0082) : GUIDE D'UTILISATION DE L'OBJET BIM : (19pt, blanc, gras, caps)
    - Nom famille : Parisine Office, 28pt, souligné, non-gras, caps
    - PROJET BIMLIB : Parisine Office, 28pt, smallCaps
    - BIBLIOTHÈQUE BIM RATP : Parisine Office, 28pt, smallCaps
    - Tableau Validation / Approbation (5 colonnes, en-tête gris C0C0C0, 9pt)
    - Tableau Suivi des modifications (5 colonnes, en-tête 25% gris, 10pt)
    """
    import base64, io as _io
    if date_str is None:
        date_str = datetime.date.today().strftime("%d/%m/%Y")

    FONT_RATP = "Parisine Office"  # Police RATP — se substitue par Arial si absente

    def run_ratp(para, texte, size_pt, bold=False, italic=False, caps=False,
                 small_caps=False, underline=False, color=None, spacing_val=None):
        """Crée un run avec police Parisine Office (fallback Arial)."""
        r = para.add_run(texte)
        rPr = r._r.get_or_add_rPr()
        # Police
        rFonts = OxmlElement('w:rFonts')
        rFonts.set(qn('w:ascii'), FONT_RATP)
        rFonts.set(qn('w:hAnsi'), FONT_RATP)
        rFonts.set(qn('w:eastAsia'), FONT_RATP)
        rFonts.set(qn('w:cs'), FONT_RATP)
        rPr.insert(0, rFonts)
        # Taille (en demi-points)
        sz = OxmlElement('w:sz'); sz.set(qn('w:val'), str(size_pt * 2)); rPr.append(sz)
        szCs = OxmlElement('w:szCs'); szCs.set(qn('w:val'), str(size_pt * 2)); rPr.append(szCs)
        # Gras
        if bold:
            b_el = OxmlElement('w:b'); rPr.append(b_el)
            bCs = OxmlElement('w:bCs'); rPr.append(bCs)
        else:
            b_el = OxmlElement('w:b'); b_el.set(qn('w:val'), '0'); rPr.append(b_el)
            bCs = OxmlElement('w:bCs'); bCs.set(qn('w:val'), '0'); rPr.append(bCs)
        # Italique
        if italic:
            i_el = OxmlElement('w:i'); rPr.append(i_el)
        # Caps
        if caps:
            caps_el = OxmlElement('w:caps'); rPr.append(caps_el)
        # SmallCaps
        if small_caps:
            sc_el = OxmlElement('w:smallCaps'); rPr.append(sc_el)
        else:
            sc_el = OxmlElement('w:smallCaps'); sc_el.set(qn('w:val'), '0'); rPr.append(sc_el)
        # Espacement lettres
        if spacing_val is not None:
            sp_el = OxmlElement('w:spacing'); sp_el.set(qn('w:val'), str(spacing_val)); rPr.append(sp_el)
        # Soulignement
        if underline:
            u_el = OxmlElement('w:u'); u_el.set(qn('w:val'), 'single'); rPr.append(u_el)
        # Couleur
        if color:
            col_el = OxmlElement('w:color'); col_el.set(qn('w:val'), color); rPr.append(col_el)
        return r

    def para_spacing(para, before=0, after=200, jc='left'):
        pPr = para._p.get_or_add_pPr()
        sp = OxmlElement('w:spacing')
        sp.set(qn('w:before'), str(before))
        sp.set(qn('w:after'), str(after))
        pPr.append(sp)
        if jc != 'left':
            jc_el = OxmlElement('w:jc'); jc_el.set(qn('w:val'), jc); pPr.append(jc_el)

    # ── Logo RATP Infrastructures ─────────────────────────────────────────────
    import base64 as _b64, io as _io
    logo_bytes = _b64.b64decode(LOGO_RATP_B64)
    p_logo = doc.add_paragraph()
    para_spacing(p_logo, before=0, after=0, jc='right')
    r_logo = p_logo.add_run()
    r_logo.add_picture(_io.BytesIO(logo_bytes), width=Cm(5.0))

    # ── Programme BIM / RATP Infrastructures / DI3D_ISI_BIM ─────────────────
    # Style BIM-CEB-TitreDépartement : Parisine Office, 12pt, after=0
    for ligne_prog in ["Programme BIM", "RATP Infrastructures", "DI3D_ISI_BIM"]:
        p = doc.add_paragraph()
        para_spacing(p, before=0, after=0)
        run_ratp(p, ligne_prog, size_pt=12, bold=False, color="000000")

    # ── Bandeau bleu nuit : GUIDE D'UTILISATION DE L'OBJET BIM ───────────────
    # Style BIM-CEB-TItreTypeDoc : fill 0A0082, texte blanc, gras, caps, 19pt
    p_bandeau = doc.add_paragraph()
    pPr_b = p_bandeau._p.get_or_add_pPr()
    shd_b = OxmlElement('w:shd')
    shd_b.set(qn('w:val'), 'clear')
    shd_b.set(qn('w:color'), 'auto')
    shd_b.set(qn('w:fill'), '0A0082')
    pPr_b.insert(0, shd_b)
    sp_b = OxmlElement('w:spacing'); sp_b.set(qn('w:before'), '120'); sp_b.set(qn('w:after'), '120')
    pPr_b.append(sp_b)
    ind_b = OxmlElement('w:ind'); ind_b.set(qn('w:left'), '120'); pPr_b.append(ind_b)
    run_ratp(p_bandeau,
             "Guide d\u2019utilisation de L\u2019objet bim : ",
             size_pt=19, bold=True, caps=True, color="FFFFFF")

    # ── Nom de la famille : BIM-CEB-TitreDocument ─────────────────────────────
    # Parisine Office, 28pt, souligné, NON gras, caps, spacing -20
    p_nom = doc.add_paragraph()
    pPr_n = p_nom._p.get_or_add_pPr()
    sp_n = OxmlElement('w:spacing'); sp_n.set(qn('w:before'), '0'); sp_n.set(qn('w:after'), '60')
    pPr_n.append(sp_n)
    jc_n = OxmlElement('w:jc'); jc_n.set(qn('w:val'), 'left'); pPr_n.append(jc_n)
    run_ratp(p_nom, nom, size_pt=28, bold=False, caps=True,
             underline=True, color="000000", spacing_val=-20)

    # ── PROJET BIMLIB ─────────────────────────────────────────────────────────
    # Style BIM-CEB-TitreSousProjet : Parisine Office, 28pt, smallCaps
    p_bimlib = doc.add_paragraph()
    pPr_bl = p_bimlib._p.get_or_add_pPr()
    sp_bl = OxmlElement('w:spacing'); sp_bl.set(qn('w:before'), '0'); sp_bl.set(qn('w:after'), '60'); pPr_bl.append(sp_bl)
    run_ratp(p_bimlib, "PROJET BIMLIB", size_pt=28, bold=False, small_caps=True, spacing_val=40)

    # ── BIBLIOTHÈQUE BIM RATP ─────────────────────────────────────────────────
    p_bibl = doc.add_paragraph()
    pPr_bi = p_bibl._p.get_or_add_pPr()
    sp_bi = OxmlElement('w:spacing'); sp_bi.set(qn('w:before'), '0'); sp_bi.set(qn('w:after'), '60'); pPr_bi.append(sp_bi)
    run_ratp(p_bibl, "BIBLIOTHEQUE BIM RATP", size_pt=28, bold=False, small_caps=True, spacing_val=40)

    # ── Tableau Validation / Approbation ─────────────────────────────────────
    # Conforme au template : 5 cols, en-tête C0C0C0, 9pt, data 9pt
    p_h1 = doc.add_paragraph()
    para_spacing(p_h1, before=120, after=0)
    rPr_h1 = OxmlElement('w:rPr')
    b_h1 = OxmlElement('w:b'); bCs_h1 = OxmlElement('w:bCs')
    caps_h1 = OxmlElement('w:caps')
    rPr_h1.append(b_h1); rPr_h1.append(bCs_h1); rPr_h1.append(caps_h1)
    p_h1._p.get_or_add_pPr().append(rPr_h1)
    r_h1 = p_h1.add_run("Validation / Approbation")
    r_h1.bold = True
    r_h1.font.size = Pt(10)

    # Colonnes template : 1346, 2126, 1843, 2126, 1418 dxa
    COL_VALID = [1346, 2126, 1843, 2126, 1418]

    def make_tbl_valid(doc, cols_dxa, headers, rows_data, hdr_fill="C0C0C0", data_sz=18):
        """Crée un tableau conforme au template (bordures simples, en-tête gris)."""
        tbl = OxmlElement('w:tbl')
        tblPr = OxmlElement('w:tblPr')
        tblW = OxmlElement('w:tblW'); tblW.set(qn('w:w'), '0'); tblW.set(qn('w:type'), 'auto')
        tblInd = OxmlElement('w:tblInd'); tblInd.set(qn('w:w'), '38'); tblInd.set(qn('w:type'), 'dxa')
        border = {'style': 'single', 'sz': '4', 'space': '0', 'color': 'auto'}
        tblBorders = OxmlElement('w:tblBorders')
        for side in ['top', 'left', 'bottom', 'right', 'insideH', 'insideV']:
            b = OxmlElement(f'w:{side}')
            b.set(qn('w:val'), border['style']); b.set(qn('w:sz'), border['sz'])
            b.set(qn('w:space'), border['space']); b.set(qn('w:color'), border['color'])
            tblBorders.append(b)
        tblLayout = OxmlElement('w:tblLayout'); tblLayout.set(qn('w:type'), 'fixed')
        tblLook = OxmlElement('w:tblLook'); tblLook.set(qn('w:val'), '01E0')
        tblLook.set(qn('w:firstRow'), '1'); tblLook.set(qn('w:lastRow'), '1')
        tblLook.set(qn('w:firstColumn'), '1'); tblLook.set(qn('w:lastColumn'), '1')
        tblLook.set(qn('w:noHBand'), '0'); tblLook.set(qn('w:noVBand'), '0')
        for el in [tblW, tblInd, tblBorders, tblLayout, tblLook]:
            tblPr.append(el)
        tbl.append(tblPr)
        tblGrid = OxmlElement('w:tblGrid')
        for w in cols_dxa:
            gc = OxmlElement('w:gridCol'); gc.set(qn('w:w'), str(w)); tblGrid.append(gc)
        tbl.append(tblGrid)

        def make_row(vals, is_header=False):
            tr = OxmlElement('w:tr')
            if not is_header:
                trPr = OxmlElement('w:trPr')
                trH = OxmlElement('w:trHeight'); trH.set(qn('w:val'), '20')
                trPr.append(trH); tr.append(trPr)
            for i, (val, w) in enumerate(zip(vals, cols_dxa)):
                tc = OxmlElement('w:tc')
                tcPr = OxmlElement('w:tcPr')
                tcW = OxmlElement('w:tcW'); tcW.set(qn('w:w'), str(w)); tcW.set(qn('w:type'), 'dxa')
                tcPr.append(tcW)
                if is_header:
                    shd = OxmlElement('w:shd'); shd.set(qn('w:val'), 'clear')
                    shd.set(qn('w:color'), 'auto'); shd.set(qn('w:fill'), hdr_fill)
                    tcPr.append(shd)
                tc.append(tcPr)
                p = OxmlElement('w:p')
                pPr = OxmlElement('w:pPr')
                sp_p = OxmlElement('w:spacing'); sp_p.set(qn('w:line'), '276'); sp_p.set(qn('w:lineRule'), 'auto')
                pPr.append(sp_p)
                if i > 0:
                    jc_p = OxmlElement('w:jc'); jc_p.set(qn('w:val'), 'center'); pPr.append(jc_p)
                p.append(pPr)
                r = OxmlElement('w:r')
                rPr = OxmlElement('w:rPr')
                if is_header:
                    b_r = OxmlElement('w:b'); rPr.append(b_r)
                sz_r = OxmlElement('w:sz')
                sz_r.set(qn('w:val'), '18' if is_header else str(data_sz))
                szCs_r = OxmlElement('w:szCs')
                szCs_r.set(qn('w:val'), '18' if is_header else str(data_sz))
                rPr.append(sz_r); rPr.append(szCs_r)
                r.append(rPr)
                t = OxmlElement('w:t'); t.set(qn('xml:space'), 'preserve'); t.text = str(val) if val else ''
                r.append(t); p.append(r)
                tc.append(p); tr.append(tc)
            return tr

        tbl.append(make_row(headers, is_header=True))
        for row_vals in rows_data:
            tbl.append(make_row(row_vals, is_header=False))
        return tbl

    # Insérer tableau Validation
    # -- Collaborateur metier selon la discipline (prefixe du nom de famille) --
    # Le nom de la famille commence par un code discipline (MEN-, CFO-, SIQ-).
    # On en deduit le collaborateur metier responsable de la validation.
    COLLABORATEURS_METIER = {
        "CFA":     "BERTHIER DE GRANDRY Foulques",
        "TLC":     "BERTHIER DE GRANDRY Foulques",
        "SIQ":     "MACHADO CARVALHO Pedro",
        "MEN":     "UK Keila",
        "COM":     "UK Keila",
        "MTS":     "UK Keila",
        "MRF":     "MONNEAU Nathalie",
        "ARC":     "CAMPOS Alexis",
        "ARC_AME": "CAMPOS Alexis",
        "CFO":     "Khaled Ben Othman",
        "FLU":     "Vincent Auger",
    }
    prefixe_disc = str(nom).split("-")[0].strip().upper() if nom else ""
    nom_collaborateur = (
        COLLABORATEURS_METIER.get(prefixe_disc)
        or COLLABORATEURS_METIER.get(prefixe_disc.split("_")[0])
        or "(à compléter)"
    )

    tbl_valid = make_tbl_valid(
        doc,
        cols_dxa=COL_VALID,
        headers=["D\u00e9partement / unit\u00e9", "Fonction", "Nom", "Visa", "Date"],
        rows_data=[
            ["GDIOITARC", "Collaborateur d\u2019architecte", nom_collaborateur, "Approuv\u00e9", date_str],
            ["", "", "", "", ""],
        ],
        hdr_fill="C0C0C0"
    )
    # Insérer via un paragraphe wrapper pour rester dans le flux courant du document
    p_valid_anchor = doc.add_paragraph()
    p_valid_anchor._p.addprevious(tbl_valid)
    p_valid_anchor._element.getparent().remove(p_valid_anchor._element)

    # ── Tableau Suivi des modifications ──────────────────────────────────────
    # Colonnes template : 851, 1134, 2996, 1418, 2551 dxa
    COL_SUIVI = [851, 1134, 2996, 1418, 2551]

    p_h2 = doc.add_paragraph()
    para_spacing(p_h2, before=120, after=0)
    r_h2 = p_h2.add_run("Suivi des modifications")
    r_h2.bold = True
    r_h2.font.size = Pt(10)

    def make_tbl_suivi(doc, cols_dxa, headers, rows_data):
        """Tableau Suivi des modifications conforme au template."""
        tbl = OxmlElement('w:tbl')
        tblPr = OxmlElement('w:tblPr')
        tblW = OxmlElement('w:tblW'); tblW.set(qn('w:w'), str(sum(cols_dxa))); tblW.set(qn('w:type'), 'dxa')
        tblLayout = OxmlElement('w:tblLayout'); tblLayout.set(qn('w:type'), 'fixed')
        tblCellMar = OxmlElement('w:tblCellMar')
        left_m = OxmlElement('w:left'); left_m.set(qn('w:w'), '79'); left_m.set(qn('w:type'), 'dxa')
        right_m = OxmlElement('w:right'); right_m.set(qn('w:w'), '79'); right_m.set(qn('w:type'), 'dxa')
        tblCellMar.append(left_m); tblCellMar.append(right_m)
        tblLook = OxmlElement('w:tblLook'); tblLook.set(qn('w:val'), '0000')
        for el in [tblW, tblLayout, tblCellMar, tblLook]:
            tblPr.append(el)
        tbl.append(tblPr)
        tblGrid = OxmlElement('w:tblGrid')
        for w in cols_dxa:
            gc = OxmlElement('w:gridCol'); gc.set(qn('w:w'), str(w)); tblGrid.append(gc)
        tbl.append(tblGrid)

        def make_row(vals, is_header=False):
            tr = OxmlElement('w:tr')
            trPr = OxmlElement('w:trPr')
            cantSplit = OxmlElement('w:cantSplit'); trPr.append(cantSplit)
            tr.append(trPr)
            for i, (val, w) in enumerate(zip(vals, cols_dxa)):
                tc = OxmlElement('w:tc')
                tcPr = OxmlElement('w:tcPr')
                tcW = OxmlElement('w:tcW'); tcW.set(qn('w:w'), str(w)); tcW.set(qn('w:type'), 'dxa')
                tcPr.append(tcW)
                tcBorders = OxmlElement('w:tcBorders')
                top_sz = '6' if is_header else '4'
                for side in ['top', 'left', 'bottom', 'right']:
                    b = OxmlElement(f'w:{side}')
                    b.set(qn('w:val'), 'single')
                    b.set(qn('w:sz'), top_sz if (side in ['top', 'left', 'right'] and is_header) else '4')
                    b.set(qn('w:space'), '0'); b.set(qn('w:color'), 'auto')
                    tcBorders.append(b)
                tcPr.append(tcBorders)
                if is_header:
                    shd = OxmlElement('w:shd'); shd.set(qn('w:val'), 'pct25')
                    shd.set(qn('w:color'), 'auto'); shd.set(qn('w:fill'), 'auto')
                    tcPr.append(shd)
                tc.append(tcPr)
                p = OxmlElement('w:p')
                pPr = OxmlElement('w:pPr')
                sp_p = OxmlElement('w:spacing'); sp_p.set(qn('w:line'), '276'); sp_p.set(qn('w:lineRule'), 'auto')
                pPr.append(sp_p)
                jc_p = OxmlElement('w:jc'); jc_p.set(qn('w:val'), 'center'); pPr.append(jc_p)
                p.append(pPr)
                r = OxmlElement('w:r')
                rPr = OxmlElement('w:rPr')
                if is_header:
                    b_r = OxmlElement('w:b'); rPr.append(b_r)
                sz_r = OxmlElement('w:sz'); sz_r.set(qn('w:val'), '20'); rPr.append(sz_r)
                szCs_r = OxmlElement('w:szCs'); szCs_r.set(qn('w:val'), '20'); rPr.append(szCs_r)
                r.append(rPr)
                t = OxmlElement('w:t'); t.set(qn('xml:space'), 'preserve'); t.text = str(val) if val else ''
                r.append(t); p.append(r)
                tc.append(p); tr.append(tc)
            return tr

        tbl.append(make_row(headers, is_header=True))
        for row_vals in rows_data:
            tbl.append(make_row(row_vals, is_header=False))
        return tbl

    tbl_suivi = make_tbl_suivi(
        doc,
        cols_dxa=COL_SUIVI,
        headers=["Version", "Date", "Objet de la modification", "Chapitres", "Auteur"],
        rows_data=[
            ["A", date_str, "Cr\u00e9ation", "Tous", ""],
            ["", "", "", "", ""],
        ]
    )
    p_suivi_anchor = doc.add_paragraph()
    p_suivi_anchor._p.addprevious(tbl_suivi)
    p_suivi_anchor._element.getparent().remove(p_suivi_anchor._element)

    doc.add_page_break()



# ════════════════════════════════════════════════════════════════════════════
#  SCHÉMA SVG — multi-niveaux d'imbrication
# ════════════════════════════════════════════════════════════════════════════
def generer_svg(nom_famille, types_yaml, params_yaml, familles_excel):
    """
    Logigramme SVG V7 — style épuré RATP :
    • Palette neutre (gris anthracite, pas de bleu dominant)
    • Lignes flexibles SANS tête de flèche (trait simple)
    • Annotations courtes et lisibles, positionnées proprement
    • PAS de légende (supprimée)
    • Famille hôte : rectangle à coins arrondis, fond gris foncé
    • Familles imbriquées : rectangles simples à coins arrondis
    • Types : rectangles pointillés sous la famille hôte
    """
    import textwrap

    noms_types    = [t.strip() for t in types_yaml.keys() if t and t.strip()]
    CAT_EXCLUES   = ["Symboles d'annotation", "Symboles annotatifs",
                     "Extremites des lignes de coupe", "Marqueurs de niveau"]
    familles_imbr = [f for f in familles_excel if f.get('categorie','') not in CAT_EXCLUES]

    # ── Palette RATP (rouge + gris) ──────────────────────────────────────────
    C_FOND_HOTE  = "#FFFFFF"   # fond blanc (double contour rouge)
    C_BORD_HOTE  = "#C00000"   # bordure rouge RATP
    C_TXT_HOTE   = "#C00000"   # texte rouge (fond blanc)
    C_FOND_NIV1  = "#FFF5F5"   # fond très clair pour niv1
    C_BORD_NIV1  = "#C00000"   # bordure rouge niv1
    C_TXT_NIV1   = "#C00000"   # texte rouge niv1
    C_FOND_NIV2  = "#F8F8F8"   # fond blanc cassé niv2+
    C_BORD_NIV2  = "#888888"   # bordure gris clair niv2+
    C_TXT_NIV2   = "#444444"   # texte gris foncé niv2+
    C_LIGNE      = "#AAAAAA"   # couleur des lignes de connexion (sans tête)
    C_TYPE_BORD  = "#C00000"   # bordure pointillée des types (rouge)
    C_TYPE_TXT   = "#C00000"   # texte des types (rouge)
    C_HDR_TXT    = "#C00000"   # entêtes de colonnes (rouge RATP)
    FONT         = "Segoe UI, Arial, sans-serif"

    def couleur_box(niv):
        return C_BORD_NIV1 if niv == 1 else C_BORD_NIV2

    def couleur_txt(niv):
        return C_TXT_NIV1 if niv == 1 else C_TXT_NIV2

    def fond_box(niv):
        return C_FOND_NIV1 if niv == 1 else C_FOND_NIV2

    # ── Dimensions ───────────────────────────────────────────────────────────
    BOX_W     = 170
    BOX_H     = 56
    BOX_GAP   = 18
    TYPE_W    = 180
    TYPE_H    = 44
    TYPE_GAP  = 18
    HOTE_W    = 190
    HOTE_H    = 66
    HEADER_H  = 48
    PAD_TOP   = 16
    PAD_LEFT  = 30
    COL_GAP   = 56

    max_niv = max((f.get('niveau', 1) for f in familles_imbr), default=1) if familles_imbr else 1

    HOTE_COL_W  = max(HOTE_W, TYPE_W) + 28
    IMBR_COL_W  = BOX_W + COL_GAP

    x_hote  = PAD_LEFT + HOTE_COL_W // 2
    def x_col(niv):
        return PAD_LEFT + HOTE_COL_W + (niv - 1) * IMBR_COL_W + BOX_W // 2 + COL_GAP // 2

    w_svg = PAD_LEFT + HOTE_COL_W + max_niv * IMBR_COL_W + PAD_LEFT

    # ── Regrouper par niveau ──────────────────────────────────────────────────
    par_niveau = {}
    fam_by_nom = {}
    for f in familles_imbr:
        niv = f.get('niveau', 1)
        par_niveau.setdefault(niv, []).append(f)
        fam_by_nom[f['nom']] = f

    # ── Positions Y ──────────────────────────────────────────────────────────
    def count_leaves(nom_f):
        f_ = fam_by_nom.get(nom_f)
        if not f_ or not f_.get('enfants'): return 1
        ch = [e for e in f_.get('enfants', []) if e in fam_by_nom]
        return sum(count_leaves(e) for e in ch) if ch else 1

    y_schema_top = PAD_TOP + HEADER_H
    y_positions  = {}

    def place_subtree(nom_f, y_top):
        y_c = y_top
        f_  = fam_by_nom.get(nom_f)
        if not f_: return
        for e_nom in [e for e in f_.get('enfants', []) if e in fam_by_nom]:
            slot = count_leaves(e_nom) * (BOX_H + BOX_GAP) - BOX_GAP
            y_positions[e_nom] = y_c + slot // 2
            place_subtree(e_nom, y_c)
            y_c += slot + BOX_GAP

    y_cur = y_schema_top + 28
    for f in par_niveau.get(1, []):
        slot = count_leaves(f['nom']) * (BOX_H + BOX_GAP) - BOX_GAP
        y_positions[f['nom']] = y_cur + slot // 2
        place_subtree(f['nom'], y_cur)
        y_cur += slot + BOX_GAP

    y_boxes_bottom = (max(y_positions.values()) + BOX_H // 2 + 10) if y_positions else (y_schema_top + 80)
    y_types_bottom = (y_schema_top + HOTE_H + 20 + len(noms_types) * (TYPE_H + TYPE_GAP) + 20)
    y_content_bottom = max(y_boxes_bottom, y_types_bottom)
    h_svg = y_content_bottom + 20  # pas de légende → plus compact

    # ── Helpers ───────────────────────────────────────────────────────────────
    def esc(t):
        return str(t).replace("&","&amp;").replace("<","&lt;").replace(">","&gt;")

    def wrap_text(text, n=22):
        lines_w = textwrap.wrap(esc(str(text)), n) or [esc(str(text))[:n]]
        return lines_w[:3]

    def svg_text_lines(lines_t, cx, cy, size=10, fill="#000000", bold=False, anchor="middle"):
        w = "600" if bold else "400"
        lh = size + 4
        total_h = len(lines_t) * lh
        y0 = cy - total_h / 2 + size
        return "\n".join(
            f'<text x="{cx}" y="{y0 + i*lh}" text-anchor="{anchor}" '
            f'font-family="{FONT}" font-size="{size}" '
            f'fill="{fill}" font-weight="{w}">{l}</text>'
            for i, l in enumerate(lines_t)
        )

    parts = []

    def ligne_flexible(x1, y1, x2, y2, sw="1.2"):
        """Ligne de connexion flexible (bezier) SANS tête de flèche."""
        dx = x2 - x1
        cx1 = x1 + dx * 0.5
        cx2 = x2 - dx * 0.3
        return (
            f'<path d="M {x1},{y1} C {cx1},{y1} {cx2},{y2} {x2},{y2}" '
            f'fill="none" stroke="{C_LIGNE}" stroke-width="{sw}"/>'
        )

    # ── Fond SVG ──────────────────────────────────────────────────────────────
    parts.append(f'<rect x="0" y="0" width="{w_svg}" height="{h_svg}" fill="white" stroke="none"/>')

    # ── Entêtes de colonnes ───────────────────────────────────────────────────
    y_hdr = PAD_TOP + 22
    parts.append(svg_text_lines(["FAMILLE HÔTE"], x_hote, y_hdr, size=9, fill=C_HDR_TXT, bold=True))
    for niv in range(1, max_niv + 1):
        cx_niv = x_col(niv)
        label  = "FAMILLE IMBRIQUÉE" if niv == 1 else f"FAMILLE IMBRIQUÉE (Niv. {niv})"
        parts.append(svg_text_lines([label], cx_niv, y_hdr, size=9, fill=C_HDR_TXT, bold=True))

    # Séparateur horizontal léger
    parts.append(
        f'<line x1="{PAD_LEFT}" y1="{y_schema_top - 4}" '
        f'x2="{w_svg - PAD_LEFT}" y2="{y_schema_top - 4}" '
        f'stroke="#DDDDDD" stroke-width="0.8"/>')

    # ── Famille hôte : rectangle arrondi gris anthracite ─────────────────────
    hx  = x_hote
    hy  = y_schema_top + HOTE_H // 2 + 10
    hw2 = HOTE_W // 2

    # Rectangle hôte — fond blanc + double contour rouge (contour ext + int)
    # Contour extérieur (plus épais)
    parts.append(
        f'<rect x="{hx - hw2}" y="{hy - HOTE_H//2}" '
        f'width="{HOTE_W}" height="{HOTE_H}" rx="5" ry="5" '
        f'fill="white" stroke="{C_BORD_HOTE}" stroke-width="3"/>')
    # Contour intérieur (décalé de 4px vers l'intérieur)
    parts.append(
        f'<rect x="{hx - hw2 + 4}" y="{hy - HOTE_H//2 + 4}" '
        f'width="{HOTE_W - 8}" height="{HOTE_H - 8}" rx="3" ry="3" '
        f'fill="none" stroke="{C_BORD_HOTE}" stroke-width="1"/>')
    # Nom de la famille — texte rouge (fond blanc)
    parts.append(svg_text_lines(
        wrap_text(nom_famille, 20), hx, hy,
        size=10, fill=C_BORD_HOTE, bold=True))

    # ── Types : rectangles pointillés sous la famille hôte ───────────────────
    y_ty_start = hy + HOTE_H // 2 + 16
    tw2 = TYPE_W // 2

    for idx, nt in enumerate(noms_types):
        ty_cy = y_ty_start + idx * (TYPE_H + TYPE_GAP) + TYPE_H // 2

        # Trait vertical de liaison
        if idx == 0:
            line_y1 = hy + HOTE_H // 2
        else:
            line_y1 = y_ty_start + (idx - 1) * (TYPE_H + TYPE_GAP) + TYPE_H
        parts.append(
            f'<line x1="{hx}" y1="{line_y1}" x2="{hx}" y2="{ty_cy - TYPE_H//2}" '
            f'stroke="{C_LIGNE}" stroke-width="1" stroke-dasharray="4,3"/>')

        # Rectangle pointillé
        parts.append(
            f'<rect x="{hx - tw2}" y="{ty_cy - TYPE_H//2}" '
            f'width="{TYPE_W}" height="{TYPE_H}" rx="4" '
            f'fill="white" stroke="{C_TYPE_BORD}" stroke-width="1.2" '
            f'stroke-dasharray="6,4"/>')
        # Label "Type" supprimé — plus de confusion avec le nom
        parts.append(svg_text_lines(
            wrap_text(nt, 24), hx, ty_cy,
            size=8, fill=C_TYPE_TXT))

    # ── Ligne hote → niv1 (sans tête de flèche, une seule ligne principale) ──
    if par_niveau.get(1):
        cx1 = x_col(1)
        y_niv1_first = y_positions.get(par_niveau[1][0]['nom'], y_schema_top + 50)
        x_from = hx + hw2
        x_to   = cx1 - BOX_W // 2
        parts.append(ligne_flexible(x_from, hy, x_to, y_niv1_first, sw="1.3"))

    # ── Boites imbriquées : rectangles simples à coins arrondis ──────────────
    for fam in familles_imbr:
        nom_f  = fam['nom']
        niv    = fam.get('niveau', 1)
        col    = couleur_box(niv)
        col_t  = couleur_txt(niv)
        fond   = fond_box(niv)
        cx_b   = x_col(niv)
        cy_b   = y_positions.get(nom_f, y_schema_top + 50)
        hw_b   = BOX_W // 2
        sw_b   = "1.5" if niv == 1 else "1.0"

        # Niv1 = contour pointillé rouge (distinct du hôte plein épais)
        # Niv2+ = contour simple gris fin
        if niv == 1:
            parts.append(
                f'<rect x="{cx_b - hw_b}" y="{cy_b - BOX_H//2}" '
                f'width="{BOX_W}" height="{BOX_H}" rx="5" ry="5" '
                f'fill="white" stroke="{col}" stroke-width="1.8"/>')
        else:
            parts.append(
                f'<rect x="{cx_b - hw_b}" y="{cy_b - BOX_H//2}" '
                f'width="{BOX_W}" height="{BOX_H}" rx="5" ry="5" '
                f'fill="{fond}" stroke="{col}" stroke-width="{sw_b}"/>')

        # Petite étiquette d'annotation en haut du rectangle
        annot = "Famille imbriquée partagée" if fam.get('is_shared') else "Famille imbriquée"
        parts.append(
            f'<text x="{cx_b}" y="{cy_b - BOX_H//2 + 10}" text-anchor="middle" '
            f'font-family="{FONT}" font-size="6.5" fill="{C_BORD_NIV2}" font-style="italic">{annot}</text>')

        # Nom de la famille
        parts.append(svg_text_lines(
            wrap_text(nom_f, 22), cx_b, cy_b + 4,
            size=9, fill=col_t, bold=(niv == 1)))

        # Ligne flexible SANS tête de flèche depuis le parent
        parent_nom = fam.get('parent')
        if parent_nom and parent_nom in y_positions:
            p_niv  = niv - 1
            p_cx   = x_col(p_niv)
            p_cy   = y_positions[parent_nom]
            x_from = p_cx + BOX_W // 2
            x_to   = cx_b - hw_b
            parts.append(ligne_flexible(x_from, p_cy, x_to, cy_b, sw="1.1"))
        elif niv == 1:
            x_from = hx + hw2
            x_to   = cx_b - hw_b
            parts.append(ligne_flexible(x_from, hy, x_to, cy_b, sw="1.1"))

    # ── Légende (cadres rouges RATP) ─────────────────────────────────────────
    LEGENDE_H = 90
    leg_pad = 16
    leg_y = h_svg + 10
    h_svg = h_svg + LEGENDE_H + 20  # agrandir le SVG pour la légende

    # Fond légende
    parts.append(
        f'<rect x="{leg_pad}" y="{leg_y}" '
        f'width="{w_svg - 2*leg_pad}" height="{LEGENDE_H - 4}" '
        f'fill="#FAFAFA" stroke="{C_BORD_NIV1}" stroke-width="0.8" rx="4"/>')
    parts.append(
        f'<text x="{leg_pad + 10}" y="{leg_y + 14}" '
        f'font-family="{FONT}" font-size="8" '
        f'font-weight="700" fill="{C_HDR_TXT}">LÉGENDE</text>')

    leg_items = [("Famille hôte", "hote"), ("", "type")]
    for niv in range(1, min(max_niv, 3) + 1):
        label = f"Famille imbriquée Niv. {niv}"
        leg_items.append((label, f"niv{niv}"))
    leg_items.append(("Connexion", "arrow"))

    slot_w  = (w_svg - 2 * leg_pad - 10) // max(len(leg_items), 1)
    item_w  = min(slot_w - 10, 120)
    item_h  = 28
    item_y0 = leg_y + 22

    for idx, (label, kind) in enumerate(leg_items):
        bx = leg_pad + 8 + idx * slot_w
        by = item_y0

        # Label au-dessus (masqué si vide)
        if label:
            parts.append(
                f'<text x="{bx + item_w//2}" y="{by}" '
                f'text-anchor="middle" font-family="{FONT}" '
                f'font-size="6.5" fill="#555555">{label}</text>')

        if kind == "hote":
            # Hôte : fond blanc + double contour rouge plein épais
            parts.append(
                f'<rect x="{bx}" y="{by + 4}" width="{item_w}" height="{item_h}" '
                f'rx="4" fill="white" stroke="{C_BORD_HOTE}" stroke-width="3"/>')
            parts.append(
                f'<rect x="{bx + 4}" y="{by + 8}" width="{item_w - 8}" height="{item_h - 8}" '
                f'rx="2" fill="none" stroke="{C_BORD_HOTE}" stroke-width="1"/>')
            parts.append(
                f'<text x="{bx + item_w//2}" y="{by + 4 + item_h//2 + 3}" '
                f'text-anchor="middle" font-family="{FONT}" '
                f'font-size="7" fill="{C_BORD_HOTE}" font-weight="700">Hôte</text>')

        elif kind == "type":
            # Type : tirets rouge fin — libellé unique « Type » centré
            parts.append(
                f'<rect x="{bx}" y="{by + 4}" width="{item_w}" height="{item_h}" '
                f'rx="4" fill="white" stroke="{C_TYPE_BORD}" '
                f'stroke-width="1.2" stroke-dasharray="5,3"/>')
            parts.append(
                f'<text x="{bx + item_w//2}" y="{by + 4 + item_h//2 + 3}" '
                f'text-anchor="middle" font-family="{FONT}" '
                f'font-size="7" fill="{C_TYPE_TXT}" font-weight="700">Type</text>')

        elif kind == "arrow":
            # Connexion : ligne flexible gris
            mid_x = bx + item_w // 2
            mid_y = by + 4 + item_h // 2
            parts.append(
                f'<path d="M {bx+6},{mid_y} C {mid_x},{mid_y-7} {mid_x},{mid_y+7} {bx+item_w-6},{mid_y}" '
                f'fill="none" stroke="{C_LIGNE}" stroke-width="1.4"/>')

        else:
            niv_num = int(kind.replace("niv", ""))
            if niv_num == 1:
                # Niv1 : contour continu rouge
                parts.append(
                    f'<rect x="{bx}" y="{by + 4}" width="{item_w}" height="{item_h}" '
                    f'rx="4" fill="white" stroke="{C_BORD_NIV1}" '
                    f'stroke-width="1.8"/>')
                parts.append(
                    f'<text x="{bx + item_w//2}" y="{by + 4 + item_h//2 - 2}" '
                    f'text-anchor="middle" font-family="{FONT}" '
                    f'font-size="6" fill="#AAAAAA" font-style="italic">Famille imbriquée</text>')
                parts.append(
                    f'<text x="{bx + item_w//2}" y="{by + 4 + item_h//2 + 7}" '
                    f'text-anchor="middle" font-family="{FONT}" '
                    f'font-size="7" fill="{C_TXT_NIV1}" font-weight="600">Niv. 1</text>')
            else:
                # Niv2+ : contour gris simple
                parts.append(
                    f'<rect x="{bx}" y="{by + 4}" width="{item_w}" height="{item_h}" '
                    f'rx="4" fill="{C_FOND_NIV2}" stroke="{C_BORD_NIV2}" stroke-width="1.0"/>')
                parts.append(
                    f'<text x="{bx + item_w//2}" y="{by + 4 + item_h//2 - 2}" '
                    f'text-anchor="middle" font-family="{FONT}" '
                    f'font-size="6" fill="#AAAAAA" font-style="italic">Famille imbriquée</text>')
                parts.append(
                    f'<text x="{bx + item_w//2}" y="{by + 4 + item_h//2 + 7}" '
                    f'text-anchor="middle" font-family="{FONT}" '
                    f'font-size="7" fill="{C_TXT_NIV2}" font-weight="600">Niv. {niv_num}</text>')

    svg = (
        f'<?xml version="1.0" encoding="UTF-8"?>'
        f'<svg xmlns="http://www.w3.org/2000/svg" '
        f'width="{w_svg}" height="{h_svg}" '
        f'viewBox="0 0 {w_svg} {h_svg}">'
        + "\n".join(parts) +
        f'</svg>'
    )
    return svg, w_svg, h_svg




def ajouter_zone_image(doc, largeur_cm=14, hauteur_cm=8, label="[ IMAGE ]", couleur_bord="1F3864"):
    """Placeholder image : fond blanc, bordure colorée, texte centré. L'utilisateur colle sa capture."""
    table_img = doc.add_table(rows=1, cols=1)
    table_img.style = 'Table Grid'
    table_img.autofit = False
    cell = table_img.cell(0, 0)
    tcPr = cell._tc.get_or_add_tcPr()
    # Fond BLANC
    shd = OxmlElement('w:shd')
    shd.set(qn('w:fill'), 'FFFFFF'); shd.set(qn('w:val'), 'clear')
    tcPr.append(shd)
    # Bordure colorée
    tcBorders = OxmlElement('w:tcBorders')
    for side in ['top', 'start', 'bottom', 'end']:
        el = OxmlElement(f'w:{side}')
        el.set(qn('w:val'), 'single'); el.set(qn('w:sz'), '12')
        el.set(qn('w:space'), '0'); el.set(qn('w:color'), couleur_bord)
        tcBorders.append(el)
    tcPr.append(tcBorders)
    set_col_widths(table_img, [largeur_cm])
    tr = table_img.rows[0]._tr
    trPr_el = tr.find(qn('w:trPr'))
    if trPr_el is None:
        trPr_el = OxmlElement('w:trPr'); tr.insert(0, trPr_el)
    trH = OxmlElement('w:trHeight')
    trH.set(qn('w:val'), str(int(hauteur_cm * 567))); trH.set(qn('w:hRule'), 'exact')
    trPr_el.append(trH)
    vAl = OxmlElement('w:vAlign'); vAl.set(qn('w:val'), 'center')
    tcPr.append(vAl)
    p_img = cell.paragraphs[0]
    p_img.clear()
    p_img.alignment = WD_ALIGN_PARAGRAPH.CENTER
    pPr_img = p_img._p.get_or_add_pPr()
    sp_img = OxmlElement('w:spacing'); sp_img.set(qn('w:before'), '0'); sp_img.set(qn('w:after'), '0')
    pPr_img.append(sp_img)
    r1 = p_img.add_run("🖼  ")
    r1.font.size = Pt(16)
    r2 = p_img.add_run(label)
    r2.bold = True; r2.font.size = Pt(12); r2.font.name = "Arial"
    rr, rg, rb = int(couleur_bord[0:2],16), int(couleur_bord[2:4],16), int(couleur_bord[4:6],16)
    r2.font.color.rgb = RGBColor(rr, rg, rb)
    r3 = p_img.add_run("\n(Coller la capture ici — Ctrl+V)")
    r3.italic = True; r3.font.size = Pt(9); r3.font.name = "Arial"
    r3.font.color.rgb = RGBColor(0xAA, 0xAA, 0xAA)
    doc.add_paragraph()

# ════════════════════════════════════════════════════════════════════════════
#  IMAGES DE VUES (3D / plan / élévations) — depuis la clé `images:` du YAML
# ════════════════════════════════════════════════════════════════════════════
def _norm_txt(s):
    """Minuscules sans accents : permet de comparer 'elevé' et 'eleve'."""
    import unicodedata
    s = unicodedata.normalize('NFD', str(s or '').strip().lower())
    return ''.join(c for c in s if unicodedata.category(c) != 'Mn')

NIVEAUX_DETAIL = ('faible', 'moyen', 'eleve')

def charger_images_yaml(data, yaml_path):
    """
    Lit la clé `images:` du YAML et regroupe les images par type de vue.

    Format attendu (plugin Revit) :
        images:
          - vue: 3d | plan | elevation
            nom_vue: <légende affichée sous l'image>
            fichier: images/xxx.png             # relatif au dossier du YAML
            niveau_detail: faible|moyen|eleve   # plans uniquement, optionnel

    Retourne {vue: [ {chemin, nom_vue, niveau_detail}, ... ]} en ne gardant
    que les fichiers réellement présents sur le disque.
    """
    base_dir = os.path.dirname(os.path.abspath(yaml_path))
    par_vue = {}
    for img in (data.get('images', []) or []):
        if not isinstance(img, dict):
            continue
        vue     = _norm_txt(img.get('vue'))
        fichier = str(img.get('fichier', '')).strip()
        if not vue or not fichier:
            continue
        chemin = fichier if os.path.isabs(fichier) else os.path.join(base_dir, fichier)
        if not os.path.exists(chemin):
            print(f"    ⚠️  Image déclarée mais introuvable : {chemin}")
            continue
        par_vue.setdefault(vue, []).append({
            'chemin':        chemin,
            'nom_vue':       str(img.get('nom_vue', '')).strip(),
            'niveau_detail': _norm_txt(img.get('niveau_detail')),
        })
    return par_vue

def plan_par_niveau(vues_plan, niveau):
    """
    Retourne l'image de plan correspondant au niveau de détail demandé
    ('faible' | 'moyen' | 'eleve'). Si aucune image ne porte ce niveau, on
    retombe sur l'ordre d'apparition, puis sur la dernière disponible.
    """
    if not vues_plan:
        return None
    for img in vues_plan:
        if img.get('niveau_detail') == niveau:
            return img
    idx = NIVEAUX_DETAIL.index(niveau) if niveau in NIVEAUX_DETAIL else 0
    return vues_plan[idx] if idx < len(vues_plan) else vues_plan[-1]

def inserer_image(doc, img, largeur_cm=14):
    """Insère une image centrée, avec le nom de vue en légende dessous."""
    doc.add_picture(img['chemin'], width=Cm(largeur_cm))
    doc.paragraphs[-1].alignment = WD_ALIGN_PARAGRAPH.CENTER
    if img.get('nom_vue'):
        p_cap = doc.add_paragraph()
        p_cap.alignment = WD_ALIGN_PARAGRAPH.CENTER
        style_run(p_cap.add_run(img['nom_vue']), size=9, italic=True, color="555555")

def inserer_image_ou_zone(doc, liste_images, largeur_cm, hauteur_cm,
                          label, couleur_bord="1F3864", index=0):
    """
    Insère l'image si elle existe, sinon retombe sur le placeholder
    « Coller la capture ici » (comportement historique conservé).
    """
    if liste_images and len(liste_images) > index:
        inserer_image(doc, liste_images[index], largeur_cm)
        return True
    ajouter_zone_image(doc, largeur_cm=largeur_cm, hauteur_cm=hauteur_cm,
                       label=label, couleur_bord=couleur_bord)
    return False

def inserer_galerie_images(doc, liste_images, largeur_totale_cm=16.5, max_cols=3):
    """
    Insere plusieurs images côte à côte (une colonne par image) avec leur
    légende. Au-delà de max_cols, un nouveau tableau est créé : aucune
    image n'est perdue. Utilisé pour les élévations.
    """
    if not liste_images:
        return False
    for debut in range(0, len(liste_images), max_cols):
        lot = liste_images[debut:debut + max_cols]
        n = len(lot)
        largeur_col = largeur_totale_cm / n
        tbl = doc.add_table(rows=1, cols=n)
        tbl.style = 'Table Grid'
        tbl.autofit = False
        for i, img in enumerate(lot):
            cell = tbl.cell(0, i)
            p_i = cell.paragraphs[0]
            p_i.clear()
            p_i.alignment = WD_ALIGN_PARAGRAPH.CENTER
            p_i.add_run().add_picture(img['chemin'], width=Cm(largeur_col - 0.6))
            if img.get('nom_vue'):
                p_leg = cell.add_paragraph()
                p_leg.alignment = WD_ALIGN_PARAGRAPH.CENTER
                style_run(p_leg.add_run(img['nom_vue']), size=9, italic=True, color="555555")
        set_col_widths(tbl, [largeur_col] * n)
        doc.add_paragraph()
    return True

def generer_guide_complet(yaml_path, output_path, excel_path=None, categorie_revit=None):
    """
    Cellule 2 V6 — Structure conforme au template RATP F :
    1. FICHE OBJET BIM & ILLUSTRATION
       1.1 Fiche d'identité
       1.2 Illustration
       1.3 Structure de la famille  ← LOGIGRAMME ICI
    2. REPRÉSENTATION
       2.1 3D
       2.2 2D
       2.3 Volumes de maintenance   ← familles COM-Z*
       2.4 Placement
    3. UTILISATION DANS LE PROJET
       3.1 Matériaux                ← paramètres matériau depuis YAML
    4. PARAMÈTRES
    5. NOMENCLATURE
    6. ANNEXE
    7. LEXIQUE
       7.1 Revit
       7.2 BIM
    """
    import cairosvg

    with open(yaml_path, 'r', encoding='utf-8') as f:
        data = yaml.safe_load(f)

    nom        = os.path.splitext(os.path.basename(yaml_path))[0]
    types      = data.get('types', {}) or {}
    params     = data.get('parameters', {}) or {}
    noms_types = [t.strip() for t in types.keys() if t and t.strip()]

    # ── Images de vues (3D / plan / élévations) déclarées dans le YAML ───────
    images_vues = charger_images_yaml(data, yaml_path)
    if images_vues:
        _detail = ", ".join(f"{k}:{len(v)}" for k, v in images_vues.items())
        print(f"    🖼️  Images trouvées ({_detail})")
    else:
        print("    ℹ️  Aucune image trouvée (placeholders conservés)")

    # ── Familles imbriquées : YAML en priorité, Excel en fallback ───────────
    familles_excel = []
    nested_yaml = data.get('nested_families', []) or []
    if nested_yaml:
        familles_excel = lire_imbriquees_depuis_yaml(data)
    elif excel_path and os.path.exists(excel_path):
        familles_excel = lire_excel_compat(excel_path)
        print(f"    📊 Excel (fallback) : {len(familles_excel)} famille(s) imbriquée(s)")
    else:
        print("    ℹ️  Aucune famille imbriquée (nested_families vide dans le YAML)")

    # ── Catégorie Revit ─────────────────────────────────────────────────────
    if not categorie_revit:
        categorie_revit = str(data.get('category', '')).strip() or None
    if not categorie_revit:
        cats = [f['categorie'] for f in familles_excel
                if f['categorie']
                and "annotation" not in f['categorie'].lower()
                and "générique" not in f['categorie'].lower()
                and "modèle" not in f['categorie'].lower()]
        categorie_revit = cats[0] if cats else "(à compléter)"

    # ── Paramètres matériaux ────────────────────────────────────────────────
    params_materiaux = {
        pn: pi for pn, pi in params.items()
        if 'material' in str(pi.get('type', '')).lower()
    }
    valeurs_mat = {}
    for pname in params_materiaux:
        vals = set()
        for tv in types.values():
            v = tv.get(pname)
            if v and str(v).strip() not in ('', 'None', '0', '0.0'):
                vals.add(str(v).strip())
        valeurs_mat[pname] = sorted(vals)

    # ── Familles volume de maintenance ──────────────────────────────────────
    PATTERNS_VOL = ['volume_maintenance', 'volume maintenance',
                    'com-z', 'volumemaintenance', 'vol_maint',
                    'volume_ouverture', 'volume ouverture',
                    'volumeouverture', 'vol_ouv']
    familles_vol = [
        f for f in familles_excel
        if any(p in f['nom'].lower() for p in PATTERNS_VOL)
        or ('volume' in f['nom'].lower() and 'maint' in f['nom'].lower())
        or ('volume' in f['nom'].lower() and 'ouvert' in f['nom'].lower())
    ]

    doc = Document()

    # ════════════════════════════════════════════════════════════════════════
    # Styles Heading + TOC + docDefaults — calqués sur le fichier de référence RATP
    # ════════════════════════════════════════════════════════════════════════
    try:
        styles = doc.styles

        # ── Police par défaut du document : Parisine Office 11pt ────────────
        # On injecte les docDefaults dans le XML du document
        styles_part = doc.part.styles
        styles_el   = styles_part._element
        NS_W = 'http://schemas.openxmlformats.org/wordprocessingml/2006/main'
        WW   = lambda t: f'{{{NS_W}}}{t}'

        # Trouver ou créer docDefaults
        docDefs = styles_el.find(WW('docDefaults'))
        if docDefs is None:
            docDefs = OxmlElement('w:docDefaults')
            styles_el.insert(0, docDefs)

        rPrDef = docDefs.find(WW('rPrDefault'))
        if rPrDef is None:
            rPrDef = OxmlElement('w:rPrDefault')
            docDefs.append(rPrDef)

        rPr_dd = rPrDef.find(WW('rPr'))
        if rPr_dd is None:
            rPr_dd = OxmlElement('w:rPr')
            rPrDef.append(rPr_dd)

        # Supprimer rFonts et sz existants
        for tag in [WW('rFonts'), WW('sz'), WW('szCs')]:
            old = rPr_dd.find(tag)
            if old is not None: rPr_dd.remove(old)

        # Injecter Parisine Office comme police par défaut
        rf_dd = OxmlElement('w:rFonts')
        for a in ['ascii', 'eastAsia', 'hAnsi', 'cs']:
            rf_dd.set(qn(f'w:{a}'), 'Parisine Office')
        rPr_dd.insert(0, rf_dd)

        sz_dd = OxmlElement('w:sz');   sz_dd.set(qn('w:val'), '22'); rPr_dd.append(sz_dd)
        szcs_dd = OxmlElement('w:szCs'); szcs_dd.set(qn('w:val'), '22'); rPr_dd.append(szcs_dd)

        def _clear_rpr(rPr):
            for tag in [qn('w:rFonts'), qn('w:color'), qn('w:sz'), qn('w:szCs'),
                        qn('w:b'), qn('w:bCs'), qn('w:caps'), qn('w:smallCaps'),
                        qn('w:i'), qn('w:iCs')]:
                old = rPr.find(tag)
                if old is not None: rPr.remove(old)

        def _set_font(rPr, font='Parisine Office'):
            rf = OxmlElement('w:rFonts')
            for a in ['ascii', 'hAnsi', 'eastAsia', 'cs']:
                rf.set(qn(f'w:{a}'), font)
            rPr.insert(0, rf)

        def _clr_ppr_spacing(pPr):
            for tag in [qn('w:spacing'), qn('w:ind'), qn('w:outlineLvl')]:
                old = pPr.find(tag)
                if old is not None: pPr.remove(old)

        def _set_ppr_spacing(pPr, before=None, after=None):
            sp = OxmlElement('w:spacing')
            if before is not None: sp.set(qn('w:before'), str(before))
            if after  is not None: sp.set(qn('w:after'),  str(after))
            pPr.append(sp)

        def _set_outline(pPr, level):
            ol = OxmlElement('w:outlineLvl')
            ol.set(qn('w:val'), str(level))
            pPr.append(ol)

        def _set_indent(pPr, left=None, right=None):
            ind = OxmlElement('w:ind')
            if left  is not None: ind.set(qn('w:left'),  str(left))
            if right is not None: ind.set(qn('w:right'), str(right))
            pPr.append(ind)

        # ════════════════════════════════════════════════════════════════════
        # Styles Heading 1/2/3 — modifiés DIRECTEMENT (python-docx les utilise)
        # Calqués sur les captures : noir, Parisine Office, filet sous H1
        # H1 : 16pt, gras, caps, souligné (filet), noir, spacing 240/240
        # H2 : 12pt, gras, noir, spacing 200/80
        # H3 : 11pt, gras, noir, spacing 160/60
        # Styles TM1/TM2/TM3 pour la TDM automatique
        # ════════════════════════════════════════════════════════════════════
        import copy as _copy

        def _patch_heading(style_name, sz_val, bold=True, caps=False,
                           color='000000', before=240, after=120,
                           outline_level=0, underline=False):
            """Modifier un style Heading directement dans le XML."""
            try:
                st = styles[style_name]
            except KeyError:
                return
            el = st._element

            # ── rPr ──────────────────────────────────────────────────────────
            rPr = el.get_or_add_rPr()
            # Supprimer toutes les propriétés existantes
            for tag in [qn('w:rFonts'), qn('w:b'), qn('w:bCs'),
                        qn('w:caps'), qn('w:smallCaps'), qn('w:i'), qn('w:iCs'),
                        qn('w:color'), qn('w:sz'), qn('w:szCs'),
                        qn('w:u'), qn('w:strike'), qn('w:vanish')]:
                for old in rPr.findall(tag):
                    rPr.remove(old)
            # Police Parisine Office
            rf = OxmlElement('w:rFonts')
            for a in ['ascii', 'hAnsi', 'eastAsia', 'cs']:
                rf.set(qn(f'w:{a}'), 'Parisine Office')
            rPr.insert(0, rf)
            # Couleur
            c_el = OxmlElement('w:color'); c_el.set(qn('w:val'), color); rPr.append(c_el)
            # Taille
            sz_el = OxmlElement('w:sz'); sz_el.set(qn('w:val'), str(sz_val)); rPr.append(sz_el)
            szcs_el = OxmlElement('w:szCs'); szcs_el.set(qn('w:val'), str(sz_val)); rPr.append(szcs_el)
            # Gras
            if bold:
                rPr.append(OxmlElement('w:b'))
                rPr.append(OxmlElement('w:bCs'))
            else:
                # Désactiver explicitement le gras hérité
                b_off = OxmlElement('w:b'); b_off.set(qn('w:val'), '0'); rPr.append(b_off)
            # Caps
            if caps:
                rPr.append(OxmlElement('w:caps'))

            # ── pPr ──────────────────────────────────────────────────────────
            pPr = el.get_or_add_pPr()
            for tag in [qn('w:spacing'), qn('w:outlineLvl'),
                        qn('w:ind'), qn('w:pBdr'), qn('w:jc')]:
                for old in pPr.findall(tag):
                    pPr.remove(old)
            # Espacement
            sp = OxmlElement('w:spacing')
            sp.set(qn('w:before'), str(before))
            sp.set(qn('w:after'),  str(after))
            pPr.append(sp)
            # Niveau plan (pour TDM)
            ol = OxmlElement('w:outlineLvl'); ol.set(qn('w:val'), str(outline_level)); pPr.append(ol)
            # Filet sous le titre (H1 uniquement)
            if underline:
                pBdr = OxmlElement('w:pBdr')
                bottom_bdr = OxmlElement('w:bottom')
                bottom_bdr.set(qn('w:val'), 'single')
                bottom_bdr.set(qn('w:sz'), '4')
                bottom_bdr.set(qn('w:space'), '1')
                bottom_bdr.set(qn('w:color'), '000000')
                pBdr.append(bottom_bdr)
                pPr.append(pBdr)

        # ── Heading 1 — 16pt, gras, caps, noir, filet horizontal ─────────────
        _patch_heading('Heading 1', sz_val=32, bold=True, caps=True,
                       color='000000', before=360, after=120,
                       outline_level=0, underline=True)

        # ── Heading 2 — 12pt, gras, noir ─────────────────────────────────────
        _patch_heading('Heading 2', sz_val=24, bold=True, caps=False,
                       color='000000', before=200, after=80,
                       outline_level=1, underline=False)

        # ── Heading 3 — 11pt, gras, noir ─────────────────────────────────────
        _patch_heading('Heading 3', sz_val=22, bold=True, caps=False,
                       color='000000', before=160, after=60,
                       outline_level=2, underline=False)

        # ── Police par défaut du document : Parisine Office 11pt ────────────
        styles_part = doc.part.styles
        styles_el   = styles_part._element
        NS_W = 'http://schemas.openxmlformats.org/wordprocessingml/2006/main'
        WW   = lambda t: f'{{{NS_W}}}{t}'
        docDefs = styles_el.find(WW('docDefaults'))
        if docDefs is None:
            docDefs = OxmlElement('w:docDefaults'); styles_el.insert(0, docDefs)
        rPrDef = docDefs.find(WW('rPrDefault'))
        if rPrDef is None:
            rPrDef = OxmlElement('w:rPrDefault'); docDefs.append(rPrDef)
        rPr_dd = rPrDef.find(WW('rPr'))
        if rPr_dd is None:
            rPr_dd = OxmlElement('w:rPr'); rPrDef.append(rPr_dd)
        for tag in [WW('rFonts'), WW('sz'), WW('szCs')]:
            old = rPr_dd.find(tag)
            if old is not None: rPr_dd.remove(old)
        rf_dd = OxmlElement('w:rFonts')
        for a in ['ascii', 'eastAsia', 'hAnsi', 'cs']:
            rf_dd.set(qn(f'w:{a}'), 'Parisine Office')
        rPr_dd.insert(0, rf_dd)
        sz_dd   = OxmlElement('w:sz');   sz_dd.set(qn('w:val'), '22'); rPr_dd.append(sz_dd)
        szcs_dd = OxmlElement('w:szCs'); szcs_dd.set(qn('w:val'), '22'); rPr_dd.append(szcs_dd)

        # ── Styles TDM (TOC 1/2/3) pour la TDM automatique ──────────────────
        # Ces styles sont utilisés par le champ TOC Word
        # Calqués sur la capture image 3 :
        # TOC 1 : bold + caps + points de conduite, 10pt
        # TOC 2 : smallCaps + indent 210, 10pt
        # TOC 3 : italic + indent 420, 10pt

        def _patch_toc(style_name, sz=20, bold=False, caps=False,
                       small_caps=False, italic=False, indent_left=0, before=0, after=0):
            try:    st = styles[style_name]
            except: st = styles.add_style(style_name, 1)
            el = st._element
            rPr = el.get_or_add_rPr()
            for tag in [qn('w:rFonts'), qn('w:b'), qn('w:bCs'),
                        qn('w:caps'), qn('w:smallCaps'), qn('w:i'), qn('w:iCs'),
                        qn('w:color'), qn('w:sz'), qn('w:szCs')]:
                for old in rPr.findall(tag):
                    rPr.remove(old)
            rf = OxmlElement('w:rFonts')
            for a in ['ascii', 'hAnsi', 'eastAsia', 'cs']:
                rf.set(qn(f'w:{a}'), 'Parisine Office')
            rPr.insert(0, rf)
            if bold:
                rPr.append(OxmlElement('w:b'))
                rPr.append(OxmlElement('w:bCs'))
            if caps:   rPr.append(OxmlElement('w:caps'))
            if small_caps: rPr.append(OxmlElement('w:smallCaps'))
            if italic:
                rPr.append(OxmlElement('w:i'))
                rPr.append(OxmlElement('w:iCs'))
            sz_el = OxmlElement('w:sz'); sz_el.set(qn('w:val'), str(sz)); rPr.append(sz_el)
            szcs_el = OxmlElement('w:szCs'); szcs_el.set(qn('w:val'), str(sz)); rPr.append(szcs_el)
            pPr = el.get_or_add_pPr()
            for tag in [qn('w:spacing'), qn('w:ind'), qn('w:tabs')]:
                for old in pPr.findall(tag):
                    pPr.remove(old)
            if before or after:
                sp = OxmlElement('w:spacing')
                if before: sp.set(qn('w:before'), str(before))
                if after:  sp.set(qn('w:after'),  str(after))
                pPr.append(sp)
            if indent_left:
                ind = OxmlElement('w:ind'); ind.set(qn('w:left'), str(indent_left)); pPr.append(ind)
            # Points de conduite tab → numéro page
            tabs = OxmlElement('w:tabs')
            t_dot = OxmlElement('w:tab')
            t_dot.set(qn('w:val'), 'right')
            t_dot.set(qn('w:leader'), 'dot')
            t_dot.set(qn('w:pos'), '8783')
            tabs.append(t_dot)
            pPr.append(tabs)

        # TM1 / TOC 1 — bold + caps, 10pt, spacing 120/60
        for nm in ['TOC 1', 'toc 1', 'Toc 1', 'TM1']:
            _patch_toc(nm, sz=20, bold=True, caps=True, before=120, after=60)

        # TM2 / TOC 2 — smallCaps, 10pt, indent 210
        for nm in ['TOC 2', 'toc 2', 'Toc 2', 'TM2']:
            _patch_toc(nm, sz=20, small_caps=True, indent_left=210)

        # TM3 / TOC 3 — italic, 10pt, indent 420
        for nm in ['TOC 3', 'toc 3', 'Toc 3', 'TM3']:
            _patch_toc(nm, sz=20, italic=True, indent_left=420)




    except Exception as _e_styles:
        pass

    # Marges
    for section in doc.sections:
        section.top_margin    = Cm(2.5)
        section.bottom_margin = Cm(2.5)
        section.left_margin   = Cm(2.5)
        section.right_margin  = Cm(2.5)
        section.header_distance = Cm(1.2)
        section.footer_distance = Cm(1.2)

    # Header/footer
    section = doc.sections[0]
    sectPr  = section._sectPr
    sectPr.append(OxmlElement('w:titlePg'))
    header = section.header
    for p in list(header.paragraphs): p._element.getparent().remove(p._element)
    header._element.append(OxmlElement('w:p'))
    ajouter_pied_page_ratp(section)
    first_footer = section.first_page_footer
    for p in list(first_footer.paragraphs): p._element.getparent().remove(p._element)
    first_footer._element.append(OxmlElement('w:p'))

    # ── PAGE DE GARDE ───────────────────────────────────────────────────────
    ajouter_page_garde(doc, nom)

    # Forcer numérotation qui commence à 1 après la page de garde
    # La page de garde est en section séparée (first page header/footer)
    # Les pages suivantes démarrent à 1 automatiquement via titlePg

    # ── TABLE DES MATIERES — Champ TOC Word (génère automatiquement la TDM) ──
    # Titre "Table des matières"
    p_toc_title = doc.add_paragraph()
    pPr_tt = p_toc_title._p.get_or_add_pPr()
    sp_tt = OxmlElement('w:spacing')
    sp_tt.set(qn('w:before'), '0'); sp_tt.set(qn('w:after'), '200')
    pPr_tt.append(sp_tt)
    r_tt = p_toc_title.add_run("Table des matieres")
    rPr_tt = r_tt._r.get_or_add_rPr()
    rf_tt = OxmlElement('w:rFonts')
    for a in ['ascii', 'hAnsi', 'eastAsia', 'cs']:
        rf_tt.set(qn(f'w:{a}'), 'Parisine Office')
    rPr_tt.insert(0, rf_tt)
    c_tt   = OxmlElement('w:color');  c_tt.set(qn('w:val'),  '070061'); rPr_tt.append(c_tt)
    b_tt   = OxmlElement('w:b');      b_tt.set(qn('w:val'),  '0');      rPr_tt.append(b_tt)
    sz_tt  = OxmlElement('w:sz');     sz_tt.set(qn('w:val'), '32');     rPr_tt.append(sz_tt)
    szC_tt = OxmlElement('w:szCs');   szC_tt.set(qn('w:val'),'32');     rPr_tt.append(szC_tt)

    # Champ TOC automatique Word
    # dirty=true + updateFields dans settings → recalcul complet à l'ouverture
    # Switches : \o "1-3" niveaux 1-3 | \h hyperliens | \z masque web | \u style outline
    p_toc = doc.add_paragraph()
    pPr_toc = p_toc._p.get_or_add_pPr()
    sp_toc = OxmlElement('w:spacing')
    sp_toc.set(qn('w:before'), '0'); sp_toc.set(qn('w:after'), '0')
    pPr_toc.append(sp_toc)

    fc_begin = OxmlElement('w:fldChar')
    fc_begin.set(qn('w:fldCharType'), 'begin')
    fc_begin.set(qn('w:dirty'), 'true')

    instr = OxmlElement('w:instrText')
    instr.set(qn('xml:space'), 'preserve')
    instr.text = ' TOC \\o "1-3" \\h \\z \\u '

    fc_sep = OxmlElement('w:fldChar')
    fc_sep.set(qn('w:fldCharType'), 'separate')

    fc_end = OxmlElement('w:fldChar')
    fc_end.set(qn('w:fldCharType'), 'end')

    r_begin = OxmlElement('w:r'); r_begin.append(fc_begin)
    r_instr = OxmlElement('w:r'); r_instr.append(instr)
    r_sep   = OxmlElement('w:r'); r_sep.append(fc_sep)

    # Texte de repli affiché tant que Word n'a pas recalculé le champ TOC.
    # Il est automatiquement remplacé par le sommaire à la mise à jour.
    r_ph = OxmlElement('w:r')
    t_ph = OxmlElement('w:t'); t_ph.set(qn('xml:space'), 'preserve')
    t_ph.text = 'Sommaire — clic droit puis « Mettre à jour les champs » (ou Ctrl+A puis F9)'
    r_ph.append(t_ph)

    r_end   = OxmlElement('w:r'); r_end.append(fc_end)

    for el in [r_begin, r_instr, r_sep, r_ph, r_end]:
        p_toc._p.append(el)

    doc.add_page_break()

    # ════════════════════════════════════════════════════════════════════════
    # 1. FICHE OBJET BIM & ILLUSTRATION
    # ════════════════════════════════════════════════════════════════════════
    doc.add_heading("1.  FICHE OBJET BIM & ILLUSTRATION", 1)

    # 1.1 Fiche d'identité — DYNAMIQUE V21 (data-driven depuis le YAML)
    doc.add_heading("1.1.  Fiche d’identité", 2)

    metadonnees = extraire_metadonnees(data, nom)
    # On conserve la catégorie déjà résolue en amont (YAML → familles imbriquées
    # → fallback), plus riche que data.get('category') seul.
    if categorie_revit:
        metadonnees['categorie'] = categorie_revit

    champs_id = [
        ("Nom de l'objet",                      metadonnees['nom_objet']),
        ("Discipline",                          metadonnees['discipline']),
        ("Catégorie",                           metadonnees['categorie']),
        ("Version Revit",                       metadonnees['version_revit']),
        ("Code Uniformat II",                   metadonnees['uniformat_code']),
        ("Glossaire",                           metadonnees['glossaire']),
        ("Référence fiche Technique / produit", metadonnees['reference_produit']),
    ]

    tbl_id = doc.add_table(rows=len(champs_id), cols=2)
    tbl_id.style = 'Table Grid'
    tbl_id.autofit = False

    for i, (label, val) in enumerate(champs_id):
        row = tbl_id.rows[i]
        cellule_donnee(row.cells[0], label, bold=True)
        cellule_donnee(row.cells[1], val, pair=i % 2 == 0)

    set_col_widths(tbl_id, [5.0, 12.0])
    doc.add_paragraph()

    # 1.2 Illustration — image 3D du YAML si disponible, sinon placeholder
    doc.add_heading("1.2.  Illustration", 2)
    inserer_image_ou_zone(doc, images_vues.get('3d'), 14, 9,
                          "ILLUSTRATION — Objet BIM", "1F3864")

    # 1.3 Structure de la famille — LOGIGRAMME (titre + schéma sur une seule page)
    doc.add_heading("1.3.  Structure de la famille", 2)
    p_svg_intro = doc.add_paragraph()
    r_svg_intro = p_svg_intro.add_run(
        "La famille est construite avec plusieurs niveaux de familles imbriquées. "
        "Le schéma ci-dessous illustre la structure de modélisation :"
    )
    style_run(r_svg_intro, size=10)
    doc.add_paragraph()

    # Logigramme SVG — 1 seul exemplaire, agrandi pour lisibilité
    svg_str, svg_w, svg_h = generer_svg(nom, types, params, familles_excel)
    png_bytes = cairosvg.svg2png(bytestring=svg_str.encode('utf-8'),
                                  output_width=svg_w * 2, output_height=svg_h * 2)
    doc.add_picture(io.BytesIO(png_bytes), width=Cm(16))
    doc.paragraphs[-1].alignment = WD_ALIGN_PARAGRAPH.CENTER
    doc.add_paragraph()

    # Tableau des familles imbriquées
    if familles_excel:
        nb_niv1  = sum(1 for f in familles_excel if f.get('niveau', 1) == 1)
        nb_total = len(familles_excel)
        intro = (
            f"La famille « {nom} » contient {nb_niv1} composant(s) imbriqué(s) direct(s)"
            + (f" ({nb_total} au total, tous niveaux)" if nb_total > nb_niv1 else "")
            + " :"
        )
        p = doc.add_paragraph()
        style_run(p.add_run(intro), size=10)
        tbl_imb = doc.add_table(rows=1, cols=4)
        tbl_imb.style = 'Table Grid'
        tbl_imb.autofit = False
        entete_tableau(tbl_imb, ["Famille imbriquée", "Catégorie Revit", "Niv.", "Types disponibles"])
        for idx, fam in enumerate(familles_excel):
            row = tbl_imb.add_row()
            pair = idx % 2 == 0
            indent = "  " * (fam.get('niveau', 1) - 1)
            cellule_donnee(row.cells[0], indent + fam['nom'], pair)
            cellule_donnee(row.cells[1], fam['categorie'], pair, center=True)
            cellule_donnee(row.cells[2], str(fam.get('niveau', 1)), pair, center=True)
            types_list = fam.get('types', [])
            if not types_list and fam.get('type'): types_list = [fam['type']]
            ts = " / ".join(str(t) for t in types_list[:4])
            if len(types_list) > 4: ts += f" (+{len(types_list)-4})"
            cellule_donnee(row.cells[3], ts, pair)
        set_col_widths(tbl_imb, [5.5, 3.5, 1.2, 6.8])
        doc.add_paragraph()

    doc.add_page_break()

    # ════════════════════════════════════════════════════════════════════════
    # 2. REPRÉSENTATION
    # ════════════════════════════════════════════════════════════════════════
    doc.add_heading("2.  REPRÉSENTATION", 1)

    # 2.1 3D — toutes les vues 3D du YAML si disponibles, sinon placeholder
    doc.add_heading("2.1.  3D :", 2)
    vues_3d = images_vues.get('3d') or []
    if len(vues_3d) > 1:
        inserer_galerie_images(doc, vues_3d)
    else:
        inserer_image_ou_zone(doc, vues_3d, 14, 8,
                              "VUE 3D — Objet BIM", "1F3864")

    # 2.2 2D
    doc.add_heading("2.2.  2D :", 2)
    p_plan = doc.add_paragraph("Vue en plan dans un projet Revit :")
    style_run(p_plan.runs[0], bold=True, size=10)

    # Vue(s) en plan du YAML : une par niveau de détail (faible / moyen / élevé)
    vues_plan = images_vues.get('plan') or []
    t2d = doc.add_table(rows=2, cols=3)
    t2d.style = 'Table Grid'
    entete_tableau(t2d, ["Niveau de détail faible", "Niveau de détail moyen", "Niveau de détail élevé"])
    tr_2d = t2d.rows[1]._tr
    trPr_2d = tr_2d.find(qn('w:trPr'))
    if trPr_2d is None:
        trPr_2d = OxmlElement('w:trPr'); tr_2d.insert(0, trPr_2d)
    trH_2d = OxmlElement('w:trHeight')
    trH_2d.set(qn('w:val'), str(int(4.0 * 567)))
    # Hauteur fixe si placeholders, extensible si de vraies images sont insérées
    trH_2d.set(qn('w:hRule'), 'atLeast' if vues_plan else 'exact')
    trPr_2d.append(trH_2d)
    for idx_2d, cell in enumerate(t2d.rows[1].cells):
        tcPr_2d = cell._tc.get_or_add_tcPr()
        # Fond BLANC
        shd_2d = OxmlElement('w:shd'); shd_2d.set(qn('w:fill'), 'FFFFFF'); shd_2d.set(qn('w:val'), 'clear')
        tcPr_2d.append(shd_2d)
        vAl_2d = OxmlElement('w:vAlign'); vAl_2d.set(qn('w:val'), 'center'); tcPr_2d.append(vAl_2d)
        p_2d = cell.paragraphs[0]; p_2d.clear()
        p_2d.alignment = WD_ALIGN_PARAGRAPH.CENTER
        # 1 image de plan par colonne, appariée grâce au champ niveau_detail
        # du YAML (faible / moyen / eleve), avec repli si un niveau manque.
        img_plan = plan_par_niveau(vues_plan, NIVEAUX_DETAIL[idx_2d])
        if img_plan:
            p_2d.add_run().add_picture(img_plan['chemin'], width=Cm(4.9))
        else:
            r_2d = p_2d.add_run("🖼  Image 2D")
            r_2d.bold = True; r_2d.font.size = Pt(11); r_2d.font.name = "Arial"
            r_2d.font.color.rgb = RGBColor(0xAA, 0xAA, 0xAA)
            r_2d2 = p_2d.add_run("\n(Coller la capture — Ctrl+V)")
            r_2d2.italic = True; r_2d2.font.size = Pt(8); r_2d2.font.name = "Arial"
            r_2d2.font.color.rgb = RGBColor(0xCC, 0xCC, 0xCC)
    set_col_widths(t2d, [5.5, 5.5, 5.5])

    # 2.3 Volumes de maintenance ou d'ouverture
    doc.add_heading("2.3.  Volumes de maintenance ou d’ouverture :", 2)
    if familles_vol:
        p = doc.add_paragraph()
        style_run(p.add_run(
            f"La famille intègre {len(familles_vol)} volume(s) de maintenance "
            "sous forme de famille(s) imbriquée(s) partagée(s)."
        ), size=10)
        tbl_vol = doc.add_table(rows=1, cols=3)
        tbl_vol.style = 'Table Grid'; tbl_vol.autofit = False
        entete_tableau(tbl_vol, ["Famille volume de maintenance", "Catégorie Revit", "Partagée"])
        for idx, fv in enumerate(familles_vol):
            row = tbl_vol.add_row(); pair = idx % 2 == 0
            cellule_donnee(row.cells[0], fv['nom'], pair)
            cellule_donnee(row.cells[1], fv['categorie'], pair, center=True)
            cellule_donnee(row.cells[2], "Oui" if fv.get('is_shared') else "Non", pair, center=True)
        set_col_widths(tbl_vol, [10.0, 5.0, 2.0])
        doc.add_paragraph()
        p_note_vol = doc.add_paragraph()
        style_run(p_note_vol.add_run("Gestion de la visibilité : "), bold=True, size=9, color="1F3864")
        style_run(p_note_vol.add_run(
            "La visibilité du volume de maintenance est contrôlée par un paramètre "
            "d'occurrence de type Oui/Non (contraintes). Ce paramètre permet à chaque "
            "modeleur de l'activer ou de le désactiver indépendamment pour chaque "
            "instance placée dans le projet, sans affecter les autres instances."
        ), size=9)
    else:
        p = doc.add_paragraph()
        style_run(p.add_run(
            "Cette famille ne contient pas de volume de maintenance."
        ), size=10, italic=True, color="888888")


    doc.add_page_break()

    # ════════════════════════════════════════════════════════════════════════
    # 3. UTILISATION DANS LE PROJET
    # ════════════════════════════════════════════════════════════════════════
    doc.add_heading("3.  UTILISATION DANS LE PROJET", 1)

    # 3.1 Placement (déplacé depuis Représentation)
    doc.add_heading("3.1.  Placement :", 2)
    p = doc.add_paragraph()
    style_run(p.add_run(
        "Le placement de la famille s'effectue en vue de plan sur le niveau courant."
    ), size=10)
    doc.add_paragraph()

    # Élévations issues du YAML (Avant / Droite / Gauche…) — légendes = nom_vue
    vues_elev = images_vues.get('elevation') or []
    if vues_elev:
        p_elev = doc.add_paragraph("Élévations :")
        style_run(p_elev.runs[0], bold=True, size=10)
        inserer_galerie_images(doc, vues_elev)
    else:
        ajouter_zone_image(doc, largeur_cm=14, hauteur_cm=8,
                           label="ÉLÉVATIONS — Objet BIM", couleur_bord="1F3864")

    # 3.2 Matériaux — mise en page calquée sur le fichier de référence
    doc.add_heading("3.2.  Materiaux :", 2)

    def _titre_souligne(doc, texte, size=10):
        p = doc.add_paragraph()
        r = p.add_run(texte)
        style_run(r, size=size, bold=False)
        rPr = r._r.get_or_add_rPr()
        u = OxmlElement('w:u'); u.set(qn('w:val'), 'single'); rPr.append(u)
        return p

    def _ou_sep(doc):
        p = doc.add_paragraph()
        pPr = p._p.get_or_add_pPr()
        sp = OxmlElement('w:spacing')
        sp.set(qn('w:before'), '120'); sp.set(qn('w:after'), '60')
        pPr.append(sp)
        style_run(p.add_run("Ou"), bold=True, size=10)

    # Texte intro
    p_mat_intro = doc.add_paragraph()
    style_run(p_mat_intro.add_run(
        "Les materiaux utilises dans la famille Revit seront selectionnes par les BIM "
        "modeleurs projet, en fonction des specifications du projet et des normes "
        "definies par l'equipe."
    ), size=10)

    _ou_sep(doc)

    # Bloc 1 : Table des materiaux
    _titre_souligne(doc, "Table des materiaux")
    doc.add_paragraph()

    if params_materiaux:
        tbl_mat = doc.add_table(rows=1, cols=3)
        tbl_mat.style = 'Table Grid'; tbl_mat.autofit = False
        entete_tableau(tbl_mat, ["Parametre materiau", "Instance / Type", "Materiaux utilises"])
        for idx, (pname, pinfo) in enumerate(params_materiaux.items()):
            row = tbl_mat.add_row(); pair = idx % 2 == 0
            cellule_donnee(row.cells[0], pname, pair)
            cellule_donnee(row.cells[1],
                           "Instance" if pinfo.get('instance') else "Type",
                           pair, center=True)
            vals = valeurs_mat.get(pname, [])
            cellule_donnee(row.cells[2],
                           " / ".join(vals) if vals else "(a definir en projet)", pair)
        set_col_widths(tbl_mat, [5.5, 2.5, 9.0])
        doc.add_paragraph()
    else:
        p_no_mat = doc.add_paragraph()
        style_run(p_no_mat.add_run("(Aucun parametre materiau defini dans cette famille.)"),
                  size=9, italic=True, color="888888")
        doc.add_paragraph()

    _ou_sep(doc)

    # Bloc 2 : Cas d'utilisation (liste materiaux uniques)
    _titre_souligne(doc, "Cas d'utilisation")
    doc.add_paragraph()

    tous_materiaux = set()
    for pname in params_materiaux:
        for v in valeurs_mat.get(pname, []):
            if v and str(v).strip() not in ('', 'None', '0', '0.0'):
                tous_materiaux.add(str(v).strip())

    if tous_materiaux:
        # Une ligne par matériau : zone image carrée + nom du matériau
        # Tableau 2 colonnes : [image placeholder] | [nom matériau]
        tbl_cas = doc.add_table(rows=0, cols=2)
        tbl_cas.style = 'Table Grid'
        tbl_cas.autofit = False
        set_col_widths(tbl_cas, [1.5, 14.5])
        for idx, mat in enumerate(sorted(tous_materiaux)):
            row_cas = tbl_cas.add_row()
            pair_cas = idx % 2 == 0
            # Colonne gauche : zone image carrée (placeholder matériau)
            cell_img = row_cas.cells[0]
            couleur_cellule(cell_img, "D9E1F2" if pair_cas else "FFFFFF")
            tcPr_img = cell_img._tc.get_or_add_tcPr()
            # Hauteur de ligne fixe (1.2cm)
            trPr = row_cas._tr.get_or_add_trPr()
            trH = OxmlElement('w:trHeight')
            trH.set(qn('w:val'), str(int(1.2 * 567)))
            trH.set(qn('w:hRule'), 'exact')
            trPr.append(trH)
            # Alignement vertical centré
            vA = OxmlElement('w:vAlign'); vA.set(qn('w:val'), 'center')
            tcPr_img.append(vA)
            p_img = cell_img.paragraphs[0]
            p_img.alignment = WD_ALIGN_PARAGRAPH.CENTER
            # Cercle gris simulant l'aperçu matériau
            r_img = p_img.add_run("⬛")
            rPr_img = r_img._r.get_or_add_rPr()
            c_img = OxmlElement('w:color'); c_img.set(qn('w:val'), 'AAAAAA'); rPr_img.append(c_img)
            sz_img = OxmlElement('w:sz'); sz_img.set(qn('w:val'), '24'); rPr_img.append(sz_img)
            # Colonne droite : nom du matériau
            cell_nom = row_cas.cells[1]
            couleur_cellule(cell_nom, "D9E1F2" if pair_cas else "FFFFFF")
            tcPr_nom = cell_nom._tc.get_or_add_tcPr()
            vA2 = OxmlElement('w:vAlign'); vA2.set(qn('w:val'), 'center')
            tcPr_nom.append(vA2)
            cellule_donnee(cell_nom, mat, pair_cas)
        doc.add_paragraph()
    else:
        p_no_cas = doc.add_paragraph()
        style_run(p_no_cas.add_run(
            "(Les materiaux seront selectionnes par les BIM modeleurs projet.)"),
            size=9, italic=True, color="888888")

    doc.add_page_break()

    # ════════════════════════════════════════════════════════════════════════
    # 4. PARAMÈTRES
    # ════════════════════════════════════════════════════════════════════════
    doc.add_heading("4.  PARAMÈTRES", 1)

    groupes_params = grouper_parametres(params)
    noms_sections_params = {
        "Visibilité":             "Paramètres de Visibilité",
        "Cotes":                  "Paramètres de cotes",
        "Contraintes":            "Paramètres de contraintes",
        "Autres Paramètres":      "Autres Paramètres",
        "Général":                "Paramètres généraux",
        "Matériaux et finitions": "Matériaux et finitions",
    }
    p_intro_params = doc.add_paragraph()
    style_run(p_intro_params.add_run("Paramètres de construction de la famille"),
              bold=True, size=10, color="000000")
    p_note_params = doc.add_paragraph()
    style_run(p_note_params.add_run(
        "Il est essentiel de limiter autant que possible la modification de ces paramètres, "
        "ceux-ci étant utilisés pour définir et contraindre la géométrie."
    ), size=10)
    doc.add_paragraph()

    for nom_grp, lst in groupes_params.items():
        if not lst: continue
        # Supprimer la rubrique « Paramètres IFC » (demande guide conforme)
        if nom_grp.strip().upper() == "IFC": continue
        # Filtrer les paramètres contenant "xxxx" (insensible à la casse)
        lst_filtree = [(pn, info) for pn, info in lst if "xxxx" not in pn.lower()]
        if not lst_filtree: continue
        titre_grp = noms_sections_params.get(nom_grp, f"Paramètres de {nom_grp}")
        p_grp = doc.add_paragraph()
        style_run(p_grp.add_run(titre_grp), bold=True, size=10, color="000000")
        set_keep_with_next(p_grp)
        # 4 colonnes — nouveau template RATP :
        # Catégorie | Paramètre | Valeur (type) | Modifier (Oui/Non)
        # Pas de Formule, pas de Phases
        table = doc.add_table(rows=1, cols=4)
        table.style = 'Table Grid'; table.autofit = False
        entete_tableau(table, ["Regrouper paramètres sous", "Paramètre", "Valeur", "Modifier"])
        for idx, (pname, info) in enumerate(lst_filtree):
            row = table.add_row(); pair = idx % 2 == 0
            is_fixed = est_parametre_fixe(info)
            # 1re colonne : nom du groupe uniquement sur la 1re ligne (cf. template)
            cellule_donnee(row.cells[0], nom_grp if idx == 0 else "", pair)
            cellule_donnee(row.cells[1], pname, pair)
            cellule_donnee(row.cells[2], traduire_type(info.get('type','')), pair, center=True)
            # Modifier = "Oui" par défaut ; "Non" si formule / reporting (grisé) / type
            modifier_val = "Non" if is_fixed else "Oui"
            cellule_donnee(row.cells[3], modifier_val, pair, center=True)
        set_col_widths(table, [3.0, 7.5, 3.0, 2.5])
        doc.add_paragraph()

    doc.add_paragraph()
    # Note de légende : Modifier = "Non" → paramètre à ne pas modifier
    p_note_rouge = doc.add_paragraph()
    style_run(p_note_rouge.add_run(
        "Note : Les parametres dont la colonne Modifier indique Non ne doivent pas "
        "être modifiés. Leurs valeurs ont été fixées lors de la modélisation afin de garantir "
        "la conformité avec les différents types prévus dans le projet."
    ), size=9, italic=True, color="555555")
    doc.add_page_break()

    # ══════════════════════════════════════════════════════════════════════════
    # 5. NOMENCLATURE
    # ══════════════════════════════════════════════════════════════════════════
    doc.add_heading("5.  NOMENCLATURE", 1)
    p_nomencl = doc.add_paragraph()
    style_run(p_nomencl.add_run(
        f"Il est possible d'inclure l'objet BIM {nom} dans une nomenclature "
        "et de ressortir les différents paramètres de votre choix."
    ), size=10)
    doc.add_paragraph()
    ajouter_zone_image(doc, largeur_cm=16, hauteur_cm=9,
                       label="NOMENCLATURE — Export des paramètres", couleur_bord="888888")
    doc.add_page_break()
    # ════════════════════════════════════════════════════════════════════════
    # 6. ANNEXE — Page de référence / fiche technique produit
    # ════════════════════════════════════════════════════════════════════════
    doc.add_heading("6.  ANNEXE", 1)
    p_ann = doc.add_paragraph()
    style_run(p_ann.add_run(
        "Cette section est réservée aux fiches techniques et documents de référence "
        "associés à la famille BIM."
    ), size=10)
    doc.add_paragraph()

    # Zone dédiée à la fiche technique / référence produit
    ajouter_zone_image(doc, largeur_cm=16, hauteur_cm=20,
                       label="FICHE TECHNIQUE / RÉFÉRENCE PRODUIT", couleur_bord="888888")
    doc.add_paragraph()
    p_ann2 = doc.add_paragraph()
    style_run(p_ann2.add_run(
        "Source : Fiche technique du fabricant ou document de référence RATP."
    ), size=9, italic=True, color="888888")

    doc.add_page_break()

    # ════════════════════════════════════════════════════════════════════════
    # 7. LEXIQUE
    # ════════════════════════════════════════════════════════════════════════
    doc.add_heading("7.  LEXIQUE", 1)

    doc.add_heading("7.1.  Revit :", 2)
    # Lexique Revit contextuel : seuls les termes pertinents pour la famille
    a_imbriquees = bool(familles_excel)
    a_imbriquee_partagee = any(f.get('is_shared') for f in familles_excel)
    termes_revit = [
        ("Famille Revit",
         "Fichier avec l’extension *.rfa constituant la conception des objets dans Revit"),
    ]
    if a_imbriquees:
        termes_revit.append(("Famille imbriquée",
         "Famille insérée dans une autre famille, pour créer ainsi des familles qui contiennent "
         "une géométrie de famille combinée entre une ou plusieurs familles imbriquées"))
    if a_imbriquee_partagee:
        termes_revit.append(("Famille imbriquée partagée",
         "Même concept que famille imbriquée mais le choix du composant peut se faire à tous "
         "niveaux d’imbrication (à travers les familles hôtes et le projet par exemple)."))
    if a_imbriquees:
        termes_revit.append(("Famille hôte",
         "Une famille hébergeant d’autre(s) famille(s). Cette famille est généralement basée "
         "sur un hôte pour la mise en place (base face, niveau ou plan de référence)."))
    for terme, defi in termes_revit:
        p = doc.add_paragraph()
        pPr_lex = p._p.get_or_add_pPr()
        sp_lex = OxmlElement('w:spacing'); sp_lex.set(qn('w:before'), '0'); sp_lex.set(qn('w:after'), '60')
        pPr_lex.append(sp_lex)
        style_run(p.add_run(terme + "\n"), bold=True, size=10, color="1F3864")
        style_run(p.add_run(defi), size=10)

    doc.add_heading("7.2.  BIM", 2)
    for terme, defi in [
        ("LOD",
         "Level Of Development, ou Niveau de Développement comprend les concepts de LOd et de LOi."),
        ("LOd",
         "Le niveau de détail (Level Of Detail) correspond à la précision géométrique d’une "
         "maquette numérique, caractérisée par la précision des objets 3D qui la constituent."),
        ("LOi",
         "Le niveau d’information (Level Of Information) correspond à la richesse d’information "
         "d’une maquette numérique."),
    ]:
        p = doc.add_paragraph()
        pPr_lex2 = p._p.get_or_add_pPr()
        sp_lex2 = OxmlElement('w:spacing'); sp_lex2.set(qn('w:before'), '0'); sp_lex2.set(qn('w:after'), '60')
        pPr_lex2.append(sp_lex2)
        style_run(p.add_run(terme + "\n"), bold=True, size=10, color="1F3864")
        style_run(p.add_run(defi), size=10)

    # ── Sauvegarde ──────────────────────────────────────────────────────────
    # Forcer mise à jour auto de tous les champs (TDM, numéros de page)
    # à l'ouverture du document dans Word
    try:
        from docx.oxml.ns import qn as _qn
        settings_part = doc.settings.element
        updateFields = OxmlElement('w:updateFields')
        updateFields.set(_qn('w:val'), 'true')
        # Supprimer ancien updateFields s'il existe
        for old_uf in settings_part.findall(_qn('w:updateFields')):
            settings_part.remove(old_uf)
        settings_part.append(updateFields)
    except Exception as _e_uf:
        pass  # settings non critiques

    doc.save(output_path)

    import zipfile, shutil
    tmp_zip = output_path + '.tmp'
    with zipfile.ZipFile(output_path, 'r') as zin:
        with zipfile.ZipFile(tmp_zip, 'w', zipfile.ZIP_DEFLATED) as zout:
            for item in zin.infolist():
                data_z = zin.read(item.filename)
                if item.filename == 'word/settings.xml':
                    import re as _re
                    data_z = data_z.decode('utf-8')
                    data_z = _re.sub(
                        r'<w:zoom\b([^>]*?)/>',
                        lambda m: f'<w:zoom{m.group(1)} w:percent="100"/>' if 'percent' not in m.group(1) else m.group(0),
                        data_z)
                    data_z = data_z.encode('utf-8')
                zout.writestr(item, data_z)
    shutil.move(tmp_zip, output_path)
    print(f"  ✓ {os.path.basename(output_path)}")


print("✅ Fonctions chargées — Lance maintenant la Cellule 3 !")

✅ Fonctions chargées — Lance maintenant la Cellule 3 !


In [ ]:
from google.colab import files
import os, zipfile, shutil

# ════════════════════════════════════════════════════════════════════════════
#  CELLULE 3 V7 — Sélection des YAML + images → un guide Word par famille
#
#  ⚙️  CHOISIR LE MODE CI-DESSOUS :
#
#   MODE = "zip"     → (RECOMMANDÉ) Uploader UN fichier .zip contenant les
#                      .yaml ET le dossier images/. La structure des dossiers
#                      est conservée automatiquement : rien à ranger à la main.
#                      → Sur Windows : clic droit sur le dossier « Guide »
#                        puis « Compresser dans un fichier ZIP ».
#
#   MODE = "drive"   → Lire directement un dossier Google Drive (aucun upload).
#                      Renseigner DOSSIER_DRIVE ci-dessous.
#
#   MODE = "upload"  → Sélectionner manuellement les .yaml et les .png.
#                      Les images sont rangées automatiquement dans images/.
# ════════════════════════════════════════════════════════════════════════════

MODE          = "zip"
DOSSIER_DRIVE = "/content/drive/MyDrive/Guide"     # utilisé si MODE = "drive"

DOSSIER_TRAVAIL = '/content/uploads'
DOSSIER_GUIDES  = '/content/guides'
EXT_YAML        = ('.yaml', '.yml')
EXT_IMAGE       = ('.png', '.jpg', '.jpeg', '.bmp', '.gif', '.tif', '.tiff')


def reinitialiser_dossier(chemin):
    """Vide et recrée un dossier (évite de mélanger deux exécutions)."""
    if os.path.isdir(chemin):
        shutil.rmtree(chemin)
    os.makedirs(chemin, exist_ok=True)


def collecter_yaml(dossier):
    """Retourne {nom_famille: chemin_yaml} en parcourant récursivement dossier."""
    trouves = {}
    for racine, _dirs, fichiers in os.walk(dossier):
        for f in fichiers:
            if f.lower().endswith(EXT_YAML):
                trouves[os.path.splitext(f)[0]] = os.path.join(racine, f)
    return trouves


def compter_images(dossier):
    n = 0
    for _racine, _dirs, fichiers in os.walk(dossier):
        n += sum(1 for f in fichiers if f.lower().endswith(EXT_IMAGE))
    return n


os.makedirs(DOSSIER_GUIDES, exist_ok=True)

print("=" * 64)
print(f"  ÉTAPE 1 — Récupération des fichiers  (MODE = {MODE})")
print("=" * 64)

# ── MODE ZIP ────────────────────────────────────────────────────────────────
if MODE == "zip":
    print("  📦 Sélectionner le fichier .zip du dossier Guide")
    print("     (il doit contenir les .yaml ET le dossier images/)")
    reinitialiser_dossier(DOSSIER_TRAVAIL)
    envoi = files.upload()
    nb_zip = 0
    for nom_fichier, contenu in envoi.items():
        if not nom_fichier.lower().endswith('.zip'):
            print(f"  ⏭️  Ignoré (pas un .zip) : {nom_fichier}")
            continue
        chemin_zip = os.path.join('/content', nom_fichier)
        with open(chemin_zip, 'wb') as f:
            f.write(contenu)
        with zipfile.ZipFile(chemin_zip) as z:
            z.extractall(DOSSIER_TRAVAIL)
        nb_zip += 1
        print(f"  ✅ ZIP extrait : {nom_fichier}")
    if nb_zip == 0:
        print("❌ Aucun fichier .zip fourni.")
        raise SystemExit()
    dossier_source = DOSSIER_TRAVAIL

# ── MODE GOOGLE DRIVE ───────────────────────────────────────────────────────
elif MODE == "drive":
    from google.colab import drive
    drive.mount('/content/drive')
    dossier_source = DOSSIER_DRIVE
    print(f"  📂 Dossier lu : {dossier_source}")
    if not os.path.isdir(dossier_source):
        print(f"❌ Dossier introuvable : {dossier_source}")
        print("   Vérifier le chemin dans DOSSIER_DRIVE.")
        raise SystemExit()

# ── MODE UPLOAD MANUEL ──────────────────────────────────────────────────────
else:
    print("  📄 Sélectionner les .yaml ET les images .png (tout en une fois)")
    reinitialiser_dossier(DOSSIER_TRAVAIL)
    os.makedirs(os.path.join(DOSSIER_TRAVAIL, 'images'), exist_ok=True)
    envoi = files.upload()
    for nom_fichier, contenu in envoi.items():
        ext = os.path.splitext(nom_fichier)[1].lower()
        if ext in EXT_IMAGE:
            dest = os.path.join(DOSSIER_TRAVAIL, 'images', nom_fichier)
        elif ext in EXT_YAML:
            dest = os.path.join(DOSSIER_TRAVAIL, nom_fichier)
        else:
            print(f"  ⏭️  Ignoré (ni YAML ni image) : {nom_fichier}")
            continue
        with open(dest, 'wb') as f:
            f.write(contenu)
    dossier_source = DOSSIER_TRAVAIL

# ── Inventaire ──────────────────────────────────────────────────────────────
yaml_paths = collecter_yaml(dossier_source)
nb_images  = compter_images(dossier_source)

print()
print(f"  📄 YAML trouvés   : {len(yaml_paths)}")
print(f"  🖼️  Images trouvées : {nb_images}")
for nom in sorted(yaml_paths):
    print(f"     • {nom}")

if not yaml_paths:
    print("❌ Aucun fichier YAML trouvé. Vérifier le contenu fourni.")
    raise SystemExit()

if nb_images == 0:
    print("  ⚠️  Aucune image : les guides garderont les zones")
    print("      « Coller la capture ici » à compléter manuellement.")

# ── Étape 2 : Génération des guides ─────────────────────────────────────────
print()
print("=" * 64)
print(f"  ÉTAPE 2 — Génération des guides ({len(yaml_paths)} famille(s))")
print("=" * 64)

stats = {'ok': 0, 'erreurs': 0}
guides_generes = []

for nom_famille, yaml_path in sorted(yaml_paths.items()):
    print(f"\n  🔄 {nom_famille}")
    output_path = f'{DOSSIER_GUIDES}/Guide_{nom_famille}.docx'
    try:
        generer_guide_complet(
            yaml_path   = yaml_path,
            output_path = output_path,
        )
        guides_generes.append(output_path)
        stats['ok'] += 1
        print(f"     ✅ Guide généré : Guide_{nom_famille}.docx")
    except Exception as e:
        import traceback
        print(f"     ❌ Erreur : {e}")
        traceback.print_exc()
        stats['erreurs'] += 1

# ── Étape 3 : Zip + Téléchargement ──────────────────────────────────────────
print()
print("=" * 64)
print("  ÉTAPE 3 — Création du ZIP et téléchargement")
print("=" * 64)

if guides_generes:
    zip_path = '/content/Guides_BIM_RATP.zip'
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
        for guide_path in guides_generes:
            z.write(guide_path, os.path.basename(guide_path))
    print(f"  📦 ZIP : {len(guides_generes)} guide(s) compressé(s)")
    files.download(zip_path)
    print("  ✅ Téléchargement lancé !")
else:
    print("  ❌ Aucun guide généré.")

# ── Résumé ──────────────────────────────────────────────────────────────────
print()
print("═" * 64)
print(f"  RÉSUMÉ")
print(f"  ✅ Guides générés   : {stats['ok']}")
print(f"  ❌ Erreurs          : {stats['erreurs']}")
print(f"  📁 YAML traités     : {len(yaml_paths)}")
print(f"  🖼️  Images trouvées  : {nb_images}")
print("═" * 64)


  ÉTAPE 1 — Récupération des fichiers  (MODE = zip)
  📦 Sélectionner le fichier .zip du dossier Guide
     (il doit contenir les .yaml ET le dossier images/)


Saving REV-C3020-BandeInterception-33.zip to REV-C3020-BandeInterception-33.zip
  ✅ ZIP extrait : REV-C3020-BandeInterception-33.zip

  📄 YAML trouvés   : 11
  🖼️  Images trouvées : 75
     • MEN-B2020-PorFenExt2VanCouPoseApplique-33
     • MEN-B2020-PorFenExt2VanCouPoseTunnel-33
     • MEN-B3020-SkydomeBombe-33
     • MEN-B3020-SkydomePlat-33
     • MEN-B3020-SkydomePyramidal-33
     • MOB-C1030-BarreAppuiPMR-33
     • MOB-C1030-SeparateurUrinoir-33
     • MOB-E2010-Cendrier-33
     • MTS-C1030-EchelleCrinoline-33
     • REV-C3020-BandeEveilVigilance-33
     • REV-C3020-BandeInterception-33

  ÉTAPE 2 — Génération des guides (11 famille(s))

  🔄 MEN-B2020-PorFenExt2VanCouPoseApplique-33
    🖼️  Images trouvées (3d:1)
    🔗 YAML : 2 famille(s) imbriquée(s) (tous niveaux)
  ✓ Guide_MEN-B2020-PorFenExt2VanCouPoseApplique-33.docx
     ✅ Guide généré : Guide_MEN-B2020-PorFenExt2VanCouPoseApplique-33.docx

  🔄 MEN-B2020-PorFenExt2VanCouPoseTunnel-33
    🖼️  Images trouvées (3d:1)
    🔗 YAML

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  ✅ Téléchargement lancé !

════════════════════════════════════════════════════════════════
  RÉSUMÉ
  ✅ Guides générés   : 11
  ❌ Erreurs          : 0
  📁 YAML traités     : 11
  🖼️  Images trouvées  : 75
════════════════════════════════════════════════════════════════
